# RiskMatch project notebook

Run from the project root with a Python 3.12 kernel. Each chapter mirrors the checklist. Record real study results under their task IDs as work proceeds. Illustrative financial examples use invented inputs. The optional frontier lab requires requirements-frontier.txt. The local import cell reads an actual price snapshot when available.

The course delivery is one constrained allocation, an interactive sensitivity comparison, GitHub code, a Streamlit URL and a 15-minute client pitch. Section 3.1.4 records the local data import. The full import command is documented in [the data inventory](data/README_data.md).

## 1. Project controls and research design

In [ ]:
from pathlib import Path
import re
from math import isclose

root = Path.cwd()
plan = (root / "CHECKLIST.md").read_text()
tasks = re.findall(r"^- \[([ x])\] \*\*(\d+\.\d+\.\d+)", plan, re.M)
assert tasks and len({task for _, task in tasks}) == len(tasks)
print(f"{sum(state == 'x' for state, _ in tasks)} of {len(tasks)} implementation tasks closed")

### 1.1 Establish the course and team baseline

**Evidence to retain:** requirement mapping and environment manifest. **Course reading:** [Advanced Data Analytics Lecture3 Explique](Theory/Course_Notes/Advanced_Data_Analytics_Lecture3_Explique.md) · [Guide Ultime Chapitre 29](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_29.md).

#### 1.1.1 · QARM brief and instructor feedback

The nine-slide QARM II Projects brief by Marc-Aurèle Divernois is dated 20 August 2024. Read with `pdftotext -layout Theory/Quant_Project/Project_Brief/QARM_2_Slides___Projects.pdf -`. SHA-256 `806019b80e64e97265e0ae40fc79ac4f2e040cd637b2f26f319143d2e3843ebe`.

| Source | Course requirement | RiskMatch demonstration |
|---|---|---|
| Slide 3 and slide 4, content | Design a strategy using quantitative optimization and lecture tools for a specific client need. | Tasks 2.1.2 and 6.1.1–6.2.2 protect the dated CHF withdrawal and compare a fixed allocation with allocations under different risk limits. |
| Slide 3 and slide 4, web app | Supply a working portfolio app URL, with relevant information and interaction. | Tasks 9.1.1–9.1.2 change the withdrawal in Streamlit and reconcile the reserve and allocation with the notebook. |
| Slide 3 | Make the code available on GitHub. | Tasks 1.1.3 and 9.1.2 provide a runnable checkout and identify the release shown in the app. |
| Slides 3–4, presentation | Pitch the strategy and app to a client in 15 minutes and answer live questions. | Task 9.1.3 times the demonstration and traces an answer about costs or shortfall to its calculation. |

Slides 5–9 give possible strategies. The robust frontier is our project choice. A written report is optional. The team's dated summary of the instructor's 17 November feedback confirms the topic and 2024 brief and leaves the backtest dates and depth to the team. The source is the team's account of the reply. [Dated feedback](https://github.com/NvarMaley/TindeRisk/blob/c41fd972784a861662d1cd609df013f6d431d0ea/docs/05_DECISIONS.md#professor-feedback--2026-11-17)

#### 1.1.2 · Responsibility assignments waived

The project owner removed formal assignments and reviewer scheduling from this section on 22 November 2026. The presentation date has not been supplied.

#### 1.1.3 · Reproducible course baseline

Use CPython 3.12.6 from `.python-version`. The baseline depends only on the standard library. Optional import and solver environments have separate direct pins in `requirements-data.txt` and `requirements-frontier.txt`. Run `python -I -S tests/check_course_examples.py` from a fresh clone. The same command runs on GitHub Actions on pushes and pull requests using only the notebook and synthetic tests. Fixtures are deterministic and need no seed.

Acceptance values are CHF 15,000 reserved before a CHF 5,000 contribution settles and CHF 10,000 afterwards, minimum portfolio variance 0.008, and a 70% withdrawal-loss boundary in the independent fixture. Three tests passed in the isolated baseline. These tests cover numerical examples only.

The project grants no additional reuse licence in this setup. Imported datasets retain their source terms. The remote contains the data published by the project owner, including WRDS. Dataset redistribution rights remain unverified. The course PDF, local environment and secret files are excluded from the tracked setup.

The first CI job could not provision 3.12.6 on Ubuntu 26.04 and skipped the tests. Pinning Ubuntu 22.04 reproduced the local result. [Successful hosted run](https://github.com/NvarMaley/TindeRisk/actions/runs/45064857119).

### 1.2 Register what would change a decision

**Evidence to retain:** dated study register and reproduction log. **Course reading:** [Guide Ultime Chapitre 01](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_01.md) · [literature](Theory/Course_Notes/literature.md).

#### 1.2.1 · Create one study record per H01–H07 with paper/lab, target, horizon, universe, benchmark, variants, primary loss, useful-gain threshold and rejection rule

Frozen 22 November 2026. [Seven study records](research/studies.csv) define the hypotheses, metrics, effect thresholds and challengers. [Protocol v1](research/study_protocol.md) specifies selection, costs, multiple comparisons and fallbacks. No market performance was used to pick the numerical thresholds. Each is a project assumption.

The planning simulation prints study ID, null detection rate, power and 95% interval width in score units. It uses 10,000 draws per design, a fixed seed and an assumed stationary gain distribution. At twelve months, power to detect the registered minimum is only 0.96%–2.34% in these scenarios. All seven are exploratory at this horizon. A model missing the admission gates keeps its reference strategy.

In [ ]:
# Optional research reproduction with requirements-research.txt.
import subprocess
import sys
from importlib.util import find_spec
if all(find_spec(pkg) for pkg in ("numpy", "scipy", "pandas")):
    print(subprocess.check_output([sys.executable, "research/design_check.py"], text=True), end="")
else:
    print("Install requirements-research.txt to reproduce the recorded research output.")

H01 0.007500 0.023400 0.179895
H02 0.005200 0.014400 0.209878
H03 0.006900 0.011900 0.179895
H04 0.007200 0.009600 0.224869
H05 0.007000 0.021000 0.089948
H06 0.005700 0.020300 0.004497
H07 0.006300 0.023000 0.001499


#### 1.2.2 · Freeze development, inner validation and untouched evaluation dates after coverage audit

**Calendar decision, 22 November 2026.** For the monthly CHF frontier study use SPY, IEF and GLD with DEXSZUS (CHF per USD). Counts below are dates with a price and an observed numeric FX quote on the same calendar date. Inputs are `data/raw/etf_prices_adjusted.csv` and `data/raw/public_phase2/DEXSZUS.csv`. Execution requires separately verified availability at the order time.

| Period | Purpose | Common observations |
|---|---|---:|
| Jan 2005–Dec 2009 | Warm-up for a trailing 756-session window | 1,251 |
| Jan 2010–Dec 2015 | Develop estimators and check accounting | 1,496 |
| Jan 2016–Dec 2019 | Expanding chronological validation for selecting parameters | 994 |
| Jan 2020–Oct 2026 | Retrospective comparison across pandemic and rate shocks | 1,700 |
| Nov 2026 | Transition and protocol freeze before first prospective decision | Partial month |
| Dec 2026–Nov 2027 | Twelve prospective monthly holding periods | Pending |

Select the radius and cost rule using earlier folds and freeze before the first prospective order. At each decision use only features available by then and training outcomes released strictly earlier. Rebalance at the next eligible fill, compare on identical dates, and log missing FX or fills. Archive each forecast and target weights before observing its return. A missed start or later rule change requires a newly dated forward window. Twelve monthly outcomes support a pilot with limited precision. The [study calendar](research/study_protocol.md) freezes all seven histories and records which future observations require new data. Study-specific research eligibility remains subject to publication and execution checks.

In [ ]:
# Count coincident ETF and CHF quote dates for the registered frontier windows.
import csv
from pathlib import Path

def quoted_dates(path, fields):
    with path.open(newline="") as stream:
        rows = csv.DictReader(stream)
        date_field = rows.fieldnames[0]
        return {row[date_field] for row in rows
                if all(row[field] not in ("", ".") and float(row[field]) > 0 for field in fields)}

prices = Path("data/raw/etf_prices_adjusted.csv")
fx = Path("data/raw/public_phase2/DEXSZUS.csv")
if prices.exists() and fx.exists():
    common = quoted_dates(prices, ["SPY", "IEF", "GLD"]) & quoted_dates(fx, ["DEXSZUS"])
    for start, end in [("2005-01-01", "2009-12-31"), ("2010-01-01", "2015-12-31"),
                       ("2016-01-01", "2019-12-31"), ("2020-01-01", "2026-10-31")]:
        print(start, end, sum(start <= day <= end for day in common))
else:
    print("The coverage calculation requires both local price snapshots.")

2005-01-01 2009-12-31 1251
2010-01-01 2015-12-31 1496
2016-01-01 2019-12-31 994
2020-01-01 2026-10-31 1700


In [ ]:
from pathlib import Path
if Path("data/additional_data/manifest.csv").exists() and Path("data/raw/public_phase2/alfred/manifest.csv").exists():
    # Verify calendar boundaries against the imported manifests and lab archives.
    import csv
    from collections import defaultdict
    from pathlib import Path
    
    def read_manifest(path):
        with Path(path).open(newline="") as stream:
            return list(csv.DictReader(stream))
    
    vintages = defaultdict(list)
    for row in read_manifest("data/raw/public_phase2/alfred/manifest.csv"):
        if row["status"] == "ok":
            vintages[row["series"]].append(row["vintage"])
    for series, dates in sorted(vintages.items()):
        assert len(set(dates)) == 262
        print(series, min(dates), max(dates), len(dates), "monthly vintages")
    for row in read_manifest("data/additional_data/manifest.csv"):
        if row["provider"].startswith("WRDS"):
            print(row["dataset"], row["start_date"], row["end_date"], row["rows"], "source rows")
else:
    print("Local coverage manifests unavailable.")

CPIAUCSL 2005-01-31 2026-10-30 262 monthly vintages
INDPRO 2005-01-31 2026-10-30 262 monthly vintages
UNRATE 2005-01-31 2026-10-30 262 monthly vintages
crsp_msf 1998-01-30 2025-12-31 2639046 source rows
crsp_dsf 1998-01-02 2025-12-31 54356930 source rows
comp_funda 1995-01-31 2026-09-30 401504 source rows
ccm_lnkhist 1949-07-01 2025-12-18 32824 source rows


#### 1.2.3 · Reproduce analytical course examples for mean–variance, risk-free asset, shrinkage, Black–Litterman, risk budgets and event studies

Recomputed the numerical checkpoints with CPython 3.12.6 and the versions in requirements-research.txt. The lab notebooks are student templates. The replay adapts the earlier project implementation to the current workspace and independently recalculates the quantities from the original fixtures. SHA-256 checks identify the three local teaching data files before loading them.

| Exercise | Reference and reproduced result | Interpretation |
|---|---|---|
| LN1 mean–variance and cash | At 20% volatility, reference return 0.0786565, observed 0.0786591 (absolute tolerance 0.0001). All four tangency weights match the lab to 1e-7. | Annual decimal returns, fully invested, short positions allowed in this exercise. |
| LN2 beta | AAPL 1.03235635 and equal-weight beta 0.64995114 match to 1e-8 on 19 tickers. | OLS with an intercept relative to SPY; the dated UVXY anomaly receives the lab correction. |
| LN3 stability and shrinkage | Equal-weight volatility 0.22631176 and return 0.07794325 across 18 assets match the worksheet to 1e-8. The printed gamma 0.5070232 gives 15.0876% volatility under the closed form. Re-solving gives 0.502691 and 15.0000%. | Preserve the gamma discrepancy. Separately, a 25% diagonal-target shrink on a two-asset fixture changes cross-covariance from 0.020 to 0.015. |
| LN4 Black–Litterman | At tau 0.05, tracking error 0.01321251. For a 2% tracking-error target, tau 0.09612869 vs reference 0.09612849 (tolerance 1e-5). | Absolute and relative return views against the four-asset anchor, using the worksheet solver convention. |
| LN5 risk budgets | Equal capital weights give volatility 0.19866744. The optimized relative contributions each equal 0.25 within 1e-6. | Equal risk requires unequal capital weights. |
| LN6 independent analytical check | For two identical assets each with 20% volatility, all five fully invested splits preserve 20% volatility, with contributions proportional to weights (tolerance 1e-12). | Duplicating an exposure provides no diversification. |
| LN7 market-model events | 37,950 independently recalculated residuals match supplied values to 4.45e-16 maximum absolute error (tolerance 1e-10). Supplied event labels count 454 good, 383 bad and 294 neutral. | The rolling pre-event market model uses the previous year. Counts describe supplied flags; construction of event signals and evaluation of H06 remain in 5.1.6. |

Local lab notebook fingerprints for this replay:
- Lab - Algorithmic trading - LN7.ipynb — f43c88011d87ea42cce179cdb21347fb2e40bd179864bfef35617a3ae6e33209
- Lab - Beta - LN2.ipynb — a6b802a050ea1aeda05d151815a1e5287c117f3f5d929214e9cc65cc04a6fd7a
- Lab - Black Litterman - LN4.ipynb — 09e693eb4f20969e86f42690b5f4cb2d122057c8f911c20ae154763f1173f597
- Lab - Markowitz - LN1.ipynb — 8f707e97c4f4dfe9ca97fb62aaf4cb388de95390865e05799787ac04103d35fc
- Lab - Risk Budgeting - LN 5.ipynb — 9b7146a888d3b3f2f0bce9dca4684e0099c2f751d1f1b112547221c3f1da56b1
- Lab - Stability - LN3.ipynb — af05d41455d25ec5195d048594770de916d0e6a144483a15f2f3153f8dca3924

In [ ]:
# Optional research reproduction with requirements-research.txt.
import subprocess
import sys
from importlib.util import find_spec
from pathlib import Path
lab_data = Path("data/Data for laboratories-20261119")
inputs = ("sentratio_and_price_st.csv", "sentratio_and_price_closetoclose_adj2.pkl", "sentratio_and_price_for_pf2.pkl")
if not all((lab_data / name).is_file() for name in inputs):
    print("Course replay requires the three local teaching fixtures.")
elif all(find_spec(pkg) for pkg in ("numpy", "scipy", "pandas")):
    print(subprocess.check_output([sys.executable, "research/course_replay.py"], text=True), end="")
else:
    print("Install requirements-research.txt to reproduce the recorded research output.")

{
  "LN1": {
    "target_return": 0.0786591209421813,
    "tangency": [
      0.6363166277121641,
      0.19266965892402824,
      0.5027406043808303,
      -0.3317268910170228
    ]
  },
  "LN2": {
    "aapl": 1.0323563487803327,
    "equal_weight_beta": 0.6499511439353765
  },
  "LN3": {
    "assets": 18,
    "equal_weight_vol": 0.2263117647657292,
    "equal_weight_return": 0.07794325170626065,
    "example_gamma": 0.5026909834600884,
    "achieved_vol": 0.14999999999999994,
    "course_gamma_vol": 0.15087625266469396,
    "shrunk_cross_covariance": 0.015
  },
  "LN4": {
    "implied_phi": 1.6287422626782593,
    "tracking_at_point05": 0.013212512555321222,
    "tau_for_two_percent": 0.0961286889682836
  },
  "LN5": {
    "equal_weight_vol": 0.19866743568083822,
    "equal_weight_contributions": [
      0.06795275709748387,
      0.05544756221496079,
      0.04341426147894803,
      0.031852854889445566
    ],
    "erc": [
      0.17543859650133511,
      0.21052631570952582,
      

## 2. Profile the client

In [ ]:
# Synthetic cash mandate. No external funding, interest or other withdrawals.
capital, withdrawal, safety = 50_000, 15_000, 0
protected = withdrawal + safety
investable = max(0, capital - protected)
assert investable == 35_000 and protected + investable == capital

### 2.1 Build the factual intake

**Evidence to retain:** mandate and dated cash ladder. **Course reading:** [Portfolio-Management-explications copie](Theory/CFA/Portfolio-Management-explications%20copie.md) · [Guide Ultime Chapitre 34](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_34.md).

#### 2.1.1 · Design four screens for goal/currency/horizon, capital and contributions, withdrawals and reserves, then desired return and preferences

The four English screens in app/intake.py collect the goal and spending currency, available resources and dated deposits, payments and emergency cash, and desired return and investment constraints. Amount entries preserve exact, bounded, unknown, declined and unanswered responses. In-session snapshots retain the original answers and subsequent edits. The Streamlit AppTest corrects a withdrawal from 10,000 to 15,000 by going back through the normal UI and verifies the original and corrected amounts. In the theoretical case in 2.3.2, a possible bill triggers a review. The reserve changes only after its amount and timing are confirmed in the intake.

Run `python -m unittest discover -s tests -p 'test_intake*.py'` with requirements-app.txt. Seven tests passed on the synthetic fixtures, including the four-screen walkthrough.

#### 2.1.2 · Calculate protected and investable capital using dated withdrawals and confirmed external funding

The opening reserve equals the separate emergency buffer plus the largest positive cumulative excess of withdrawals over settled earmarked funding. When that exceeds starting capital, report the available protected amount, first shortfall date and unfunded amount. Conservative endpoints combine earlier obligations at their high amounts and confirmed resources at their low amounts. Funding due on the same day must already be settled when the bill falls due. The stress case delays confirmed external bill funding by 30 calendar days. A change to desired return or tolerance leaves the complete ladder unchanged in the fixture.

Run `python -m unittest discover -s tests -p 'test_intake*.py'` with requirements-app.txt. Seven tests passed on the synthetic fixtures, including the four-screen walkthrough.

In [ ]:
# Synthetic single-withdrawal fixture. Savings are confirmed and settle in month 12.
def review_example(phi, due_month, confirmed_saving=0):
    available_saving = confirmed_saving if due_month >= 12 else 0
    protected = max(0, 15_000 - available_saving)
    sleeve = 50_000 - protected
    # Illustrative alternatives; the annual scores apply only to this fixture.
    choices = {"stable": (0.03, 0.0025), "variable": (0.05, 0.0225)}
    fraction = sleeve / 50_000
    preferred = max(choices, key=lambda k: fraction*choices[k][0]
                    - phi*fraction**2*choices[k][1]/2)
    return protected, sleeve, preferred

tolerant = review_example(1, 24)
cautious = review_example(6, 24)
assert tolerant[:2] == cautious[:2] == (15_000, 35_000)
assert tolerant[2] != cautious[2]
assert review_example(6, 24, 5_000)[0] == 10_000
assert review_example(6, 6, 5_000)[0] == 15_000
print("Same funding, different preference:", tolerant, cautious)

#### 2.1.3 · Solve the goal-funding return from dated cash flows with an explicit timing convention

The target refers to the terminal investment goal in addition to the protected cash and scheduled bills. Cash reserved for bills earns zero. Unused earmarked cash stays reserved. Confirmed goal deposits compound from their settlement dates at a constant annual nominal rate (actual days / 365). This separation prevents double counting and leaves a monotone funding equation for nonnegative goal deposits. A target in today's money compounds at each endpoint of the stated inflation assumption. Brent's method solves the equation in log-growth space. A deposit made at maturity cannot earn a return. Missing data, insufficient reserves, unreachable targets and a solution independent of rate have explicit outcomes. The implied rate is conditional on the client's inputs.

Run `python -m unittest discover -s tests -p 'test_intake*.py'` with requirements-app.txt. Seven tests passed on the synthetic fixtures, including the four-screen walkthrough.

In [ ]:
# Reproduce the client-facing calculation with a two-year, 730-day mandate.
from importlib.util import find_spec
from pathlib import Path
from datetime import date, timedelta
import sys

if find_spec("scipy"):
    sys.path.insert(0, str(Path.cwd() / "src"))
    from riskmatch.intake import evaluate
    opening = date(2027, 1, 1)
    meeting = dict(start=opening, end=opening + timedelta(days=730), currency="CHF",
                   capital="50000", buffer="0", target="35000", desired="5",
                   bill_list="Complete", goal_list="Complete", basis="Future amount",
                   flows=[dict(kind="withdrawal", day=opening+timedelta(days=365),
                               amount="15000", confirmed=True)])
    result = evaluate(meeting)
    case = result["capacity"][1]
    print(f"Reserve CHF {case['protected']:,.0f}; investable CHF {case['investable']:,.0f}")
    print(f"Desired 5%; required {abs(result['rates'][1]['rate']):.2%} (zero to rounding)")
    real = evaluate(dict(meeting, basis="Today's purchasing power", inflation="2..4"))
    print("Inflation sensitivity:", [round(100*r["rate"], 2) for r in real["rates"]], "% nominal")
else:
    print("Install requirements-app.txt to run the mandate calculations.")

Reserve CHF 15,000; investable CHF 35,000
Desired 5%; required 0.00% (zero to rounding)
Inflation sensitivity: [2.0, 4.0] % nominal


### 2.2 Build the adaptive swipe interview

**Evidence to retain:** versioned card specification and route traces. **Course reading:** [W13 Behavioral Finance Guide FR](Theory/Course_Notes/W13_Behavioral_Finance_Guide_FR.md) · [Guide Ultime Chapitre 14](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_14.md) · [Guide Ultime Chapitre 18](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_18.md).

#### 2.2.1 · Draft 60 scenario IDs including 15 common items

Version 2.2.0 of questions/profile_v1.json supplies 60 English cards, 15 shared. The test checks the complete family blueprint, every yes/no/pass scoring key, and positivity of every numerical threshold. The theoretical review in 2.3.2 documents possible meanings, alternative explanations and the effect permitted for each family. Real-world comprehension and timing are unmeasured.

Reproduction: `python tests/test_interview.py`. Browser evidence: `python tests/browser_interview.py` against the running Streamlit app.

#### 2.2.2 · Implement one statement per gesture, right=yes and left=no, with pass, keyboard equivalent and an explanation after failed practice

Open the risk interview from the mandate review. Mouse and touch drags, arrow keys, U and three buttons share a single revision-checked event handler. The browser fixture tests short drags, pointer gestures, keyboard input, pass, stop/resume, a correction and duplicate events on desktop and mobile Chrome emulation. These checks cover desktop and emulated mobile input.

Reproduction: `python tests/test_interview.py`. Browser evidence: `python tests/browser_interview.py` against the running Streamlit app.

#### 2.2.3 · Map each risk choice to a threshold by equating the one-year mean–variance utilities of the alternatives

Only understood trade-offs supply directional constraints. The sure/uncertain crossover is twice the excess mean over the sure return divided by variance. Degenerate, dominated and nonpositive thresholds are uninformative for the nonnegative parameter domain. Report one-sided bounds, an empty intersection or inadequate evidence explicitly. CHF 1,020 against 900/1,200 from a 1,000 stake yields 8/3.

Reproduction: `python tests/test_interview.py`. Browser evidence: `python tests/browser_interview.py` against the running Streamlit app.

In [ ]:
# Equal-probability lottery, decimal one-year returns.
from statistics import fmean, pvariance
outcomes = [-0.1, 0.2]
def threshold(sure):
    return 2 * (fmean(outcomes) - sure) / pvariance(outcomes)
assert isclose(threshold(0.02), 8/3)
bounds = [threshold(0.04), threshold(0.02)]
# Consistent choices: take 4% guaranteed (lower bound), prefer gamble to 2% (upper).
assert bounds[0] < bounds[1]

#### 2.2.4 · Implement core routing, threshold probes inside uncertain intervals, paired loss comparisons and conditional follow-ups

The core route interleaves paired cards. Failed practice receives a second scenario; a second failure ends the route. Declines trigger separate questions on liquidity, discomfort and beliefs. Two numerical probes and 25 active answers bound the path. Rewinding preserves the historical log and reopens dependent choices. No-view and early-stop paths retain uncertainty.

Reproduction: `python tests/test_interview.py`. Browser evidence: `python tests/browser_interview.py` against the running Streamlit app.

#### 2.2.5 · Show tolerance range, capacity, reserve, conflicts, exclusions and monitoring preference

The report shows the hypothetical one-year bracket, confirmed financial capacity and exclusions, descriptive behavior and monitoring, and usable/passed counts. Liquidity disclosures invite the client to edit the dated schedule. Financial answers and risk attitudes enter separate calculations. The example ranking compares a stable and a variable sleeve at the stated hypothetical stake.

Reproduction: `python tests/test_interview.py`. Browser evidence: `python tests/browser_interview.py` against the running Streamlit app.

#### 2.2.6 · Propagate approximate amounts through capacity calculations

Leave out each directional trade-off in turn and recompute its compatible interval and example ranking. A single decisive choice or a contradiction can trigger one optional near-indifference clarification. Endorsing it or passing has no directional effect. Present the omission outcomes together, identify sensitive choices and defer personal ranking. Input ranges and model compatibility have distinct labels. No individual confidence level has been calibrated.

Reproduction: `python tests/test_interview.py`. Browser evidence: `python tests/browser_interview.py` against the running Streamlit app.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd() / "src"))
from riskmatch.interview import CARDS, current, eligible, report, submit, threshold
print("Bank:", len(CARDS), "cards;", sum(c["core"] for c in CARDS.values()), "common")
print("CHF 1,020 example:", round(threshold(CARDS["C03"]), 4))
for label, overrides in [("cash review", {"C12":"yes","B01":"yes"}),
                         ("discomfort", {"C12":"yes","B02":"yes"}),
                         ("unclear practice", {"C01":"pass","P01":"pass"})]:
    trace = []
    while pool := eligible(current(trace)):
        item = pool[0]
        card = CARDS[item]
        response = overrides.get(item, "yes" if item == "C01" else "no")
        if card["effect"] == "bound":
            choice = "sure" if 3 >= threshold(card) else "gamble"
            response = next(a for a,v in card["answers"].items() if v == choice)
        assert submit(trace, item, response, "fixture", len(trace))
    profile = report(current(trace))
    print(label, "→", " ".join(current(trace)), "|", profile["status"],
          "| cash review:", profile["liquidity"], "| discomfort:", profile["discomfort"])
sensitive = report({"C01":"yes","C03":"yes"})
print("One decisive response:", sensitive["sensitive"], "→", sensitive["options"],
      "or", sensitive["omissions"]["C03"][3], "when omitted")

Bank: 60 cards; 15 common
CHF 1,020 example: 2.6667
cash review → C01 C02 C03 C08 C04 C12 C05 C09 C06 C14 C10 C07 C13 C11 C15 B01 B04 B05 B06 B02 B03 B15 B09 B10 B13 | bounded | cash review: True | discomfort: False
discomfort → C01 C02 C03 C08 C04 C12 C05 C09 C06 C14 C10 C07 C13 C11 C15 B01 B02 B03 B15 B07 B09 B10 B13 | bounded | cash review: False | discomfort: True
unclear practice → C01 P01 | comprehension unresolved | cash review: False | discomfort: False
One decisive response: ['C03'] → ('stable',) or ('variable', 'stable') when omitted


In [ ]:
# An apparent band classification changes when one decisive response is removed.
choices = [("lower", 1), ("lower", 3), ("upper", 5)]
def compatible(answers):
    lo = max([v for side,v in answers if side == "lower"], default=0)
    hi = min([v for side,v in answers if side == "upper"], default=float("inf"))
    return lo, hi
assert compatible(choices) == (3, 5)
assert compatible(choices[:1] + choices[2:]) == (1, 5)
assert compatible([("lower", 5), ("upper", 2)]) == (5, 2)  # unresolved

### 2.3 Justify the questions and test the routes

The [theoretical rationale](questions/QUESTIONNAIRE_DESIGN.md#how-to-interpret-hypothetical-answers-in-the-report) covers all seven core families. The [simulation record](research/questionnaire_validation/SIMULATION.md) documents the controlled software checks.

#### 2.3.1 · First simulate complete, uncertain, contradictory and aborted routes with fixed seeds

Executed 480 simulated interviews using seed 2301, 60 for each of eight behavioral rules and 20 probe forms. All 60 cards were selected in the generated sample and have targeted reachability witnesses. See the counts per branch and one synthetic record per assignment in research/questionnaire_validation/. Reproduce with python research/simulate_interview.py and python tests/test_questionnaire_pilot.py. The invariant tests cover semantic choices across display arrangements, duplicate submissions, the first sealed core after a correction and external validation choices.

In [ ]:
# Read the recorded fixed-seed experiment. Regenerate with research/simulate_interview.py.
import json
from pathlib import Path
summary = json.loads(Path("research/questionnaire_validation/simulation_summary.json").read_text())
print("Synthetic routes:", summary["simulations"], "| cards reached:", sum(c["generated"] > 0 for c in summary["coverage"].values()))
for behavior, counts in summary["behaviors"].items():
    print(behavior, "· unresolved example ranking", counts["unresolved"], "/", counts["n"])
evaluation = summary["synthetic_evaluation"]
print("Synthetic evaluation partition:", evaluation["assigned"])
for arm in ("core", "adaptive"):
    print(arm, "prediction coverage:", f"{evaluation[arm]['coverage']:.1%}", "verified people:", evaluation[arm]["verified_people"])
print("Verified hits per assigned slot, paired change:", f"{evaluation['verified_hits_gain']['mean']:+.2%}")

Synthetic routes: 480 | cards reached: 60
consistent · unresolved example ranking 29 / 60
contradictory · unresolved example ranking 60 / 60
indifferent · unresolved example ranking 60 / 60
pass-heavy · unresolved example ranking 60 / 60
always yes · unresolved example ranking 60 / 60
random · unresolved example ranking 53 / 60
stake-dependent · unresolved example ranking 30 / 60
early stop · unresolved example ranking 41 / 60
Synthetic evaluation partition: 333
core prediction coverage: 50.5% verified people: 139
adaptive prediction coverage: 51.5% verified people: 141
Verified hits per assigned slot, paired change: +0.60%


#### 2.3.2 · What can a hypothetical answer tell us?

Suppose an investment falls 15% with eight years left. A fictional client answers “I would sell some of it” (C12). A bill due this year (B01), discomfort despite having expenses covered (B02), and an expectation of further losses (B03) would each explain the sale. The replay below changes only the endorsed reason. A review of cash needs can lead to a factual correction. The sale itself contributes no numerical tolerance bound.

In a second example the person chooses the gamble on C08 and C09 and the guarantee on C10 and C11. Both paired switches make B14 eligible. Only an explicit yes confirms that the starting balance mattered in those comparisons. Finishing wealth also changed, so the choices cannot establish a stable loss-aversion coefficient.

The 480 routes summarized above exercise the software. The worked cases illustrate interpretation. The project collected no human responses and has no measured duration, behavioral prevalence or predictive accuracy for real investors.

The [card-by-card guide](questions/GUIDE_DES_CARTES.md) lists the English questions, French interpretations and the numerical consequence of each answer. The check below reproduces its example rankings at fixed funding. Section 2.1.2 checks the effect of changing the payment date.

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path("src").resolve()))
from riskmatch.interview import CARDS, current, eligible, report, submit

def hypothetical_route(overrides):
    log = []
    while pool := eligible(current(log)):
        card = pool[0]
        default = "pass" if CARDS[card]["effect"] == "bound" else "no"
        answer = overrides.get(card, default)
        assert submit(log, card, answer, "fictional example", len(log))
    answers = current(log)
    assert overrides.items() <= answers.items()
    return report(answers)

common = {"C01": "yes", "C03": "yes", "C12": "yes"}
base = hypothetical_route(common)
assert round(base["low"], 2) == 2.67 and base["high"] is None
for card, reason, consequence in [
    ("B01", "liquidity", "Review the payment amount and date"),
    ("B02", "discomfort", "Record a preference for smaller fluctuations"),
    ("B03", "belief", "Record the expectation of further losses"),
]:
    case = hypothetical_route({**common, card: "yes"})
    assert [key for key in ("liquidity", "discomfort", "belief") if case[key]] == [reason]
    assert (case["low"], case["high"], case["used"]) == (base["low"], base["high"], 1)
    print(f"C12 sell + {card} yes → {consequence}. Same lower bound: {case['low']:.2f}.")

pair = {"C01": "yes", "C08": "no", "C10": "yes", "C09": "no", "C11": "yes"}
for response in ("yes", "pass"):
    case = hypothetical_route({**pair, "B14": response})
    assert case["loss_pattern"] and case["confirmed_loss"] == (response == "yes")
    assert case["used"] == 0
    print(f"Repeated switch + B14 {response} → reference point confirmed: {case['confirmed_loss']}.")

C12 sell + B01 yes → Review the payment amount and date. Same lower bound: 2.67.
C12 sell + B02 yes → Record a preference for smaller fluctuations. Same lower bound: 2.67.
C12 sell + B03 yes → Record the expectation of further losses. Same lower bound: 2.67.
Repeated switch + B14 yes → reference point confirmed: True.
Repeated switch + B14 pass → reference point confirmed: False.


In [ ]:
from collections import Counter
from pathlib import Path
import re
import sys

sys.path.insert(0, str(Path("src").resolve()))
from riskmatch.interview import CARDS, report, threshold

guide = Path("questions/GUIDE_DES_CARTES.md").read_text()
documented = re.findall(r"^\| ([CBPT][0-9]{2}) (?:·|\|)", guide, re.MULTILINE)
assert Counter(documented) == Counter(CARDS.keys())
quantitative = [card for card in CARDS.values() if card["effect"] == "bound"]
assert len(quantitative) == 25
for card in quantitative:
    row = next(line for line in guide.splitlines() if line.startswith(f"| {card['id']} |"))
    fields = [field.strip() for field in row.strip("|").split("|")]
    assert int("".join(fields[1].split())) == card["certain"] and fields[2] == card["statement"]
    cutoff = f"{threshold(card):.2f}".replace(".", ",")
    for response, field in zip(("yes", "no"), fields[3:], strict=True):
        label = "Garantie · φ ≥ " if card["answers"][response] == "sure" else "Pari · φ ≤ "
        assert field == label + cutoff
print("Guide verified: 60 distinct profile cards and 25 numerical yes/no mappings.")

# Unlisted trade-offs supply no directional evidence. All examples use the same mandate.
for name, answers, expected in [
    ("More stable", {"C03": "yes", "C04": "no", "C06": "yes"}, (4.44, 6.22, ("stable",), False)),
    ("More variable", {"C05": "no", "C07": "no"}, (0, 0.89, ("variable",), False)),
    ("Both possible", {"C04": "yes", "C05": "yes"}, (0.89, 4.44, ("variable", "stable"), False)),
    ("Sensitive to one choice", {"C03": "yes"}, (2.67, None, ("stable",), True)),
    ("Conflicting", {"C03": "yes", "C05": "no"}, (2.67, 0.89, (), True)),
]:
    result = report({"C01": "yes", **answers})
    high = result["high"]
    bounds = (round(result["low"], 2), None if high is None else round(high, 2))
    assert (*bounds, result["options"], bool(result["sensitive"])) == expected
    print(name, "· bounds", bounds, "·", result["status"])

Guide verified: 60 distinct profile cards and 25 numerical yes/no mappings.
More stable · bounds (4.44, 6.22) · bounded
More variable · bounds (0.0, 0.89) · one-sided
Both possible · bounds (0.89, 4.44) · bounded
Sensitive to one choice · bounds (2.67, None) · one-sided
Conflicting · bounds (2.67, 0.89) · conflict


## 3. Collect and prepare point-in-time data

In [ ]:
# An asset gains 5% in USD while CHF per USD falls 2%.
chf_return = (1 + 0.05) * (1 - 0.02) - 1
assert isclose(chf_return, 0.029)
print(f"Return in spending currency: {chf_return:.1%}")

### 3.1 Establish a source catalogue

**Evidence to retain:** series catalogue and immutable raw manifest. **Course reading:** [Guide Ultime Chapitre 08](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_08.md) · [W6 Financial Statements Analysis](Theory/Course_Notes/W6_Financial_Statements_Analysis.md).

#### Acquisition and reconciliation sheet for 3.1.1–3.2.5

Use the [local inventory](data/README_data.md) for current import status. Record actual access, vintage, rights and coverage before marking any as acquired. The public-data fixture can exercise the pipeline before vendor access is available.

| Study / source to investigate | Grain and requested fields | Preparation and admission control |
|---|---|---|
| H01/H02/H07 and benchmarks · authorized total-return provider or fund issuer | Instrument/session: stable ID, venue, close, currency, distributions, split factor, adjusted or total-return level, inception and termination. For bonds distinguish fund returns from yield changes. | Resolve venue sessions, base currency and distribution convention. Preserve dead securities when studying a historical opportunity set. Reconcile a known distribution and split against the issuer. |
| H03/H04 · FRED, ALFRED and original statistical releases | Series/vintage/observation: curve (e.g. T10Y2Y), credit spread (BAA10Y), VIXCLS, CPIAUCSL, UNRATE, INDPRO; rate level versus change, seasonal adjustment, release date and vintage bounds. | Retain initial releases and revisions separately. Derive year-on-year inflation only after enough released history. If release time is ambiguous lag conservatively; run vintage versus latest-revised sensitivity and label the latter retrospective. |
| Cash and spending currency · central bank or FRED | DTB3 is a US discount-basis bill quote; DEXSZUS quotes CHF per USD. Select a defensible CHF cash benchmark separately. Record quote convention, tenor, maturity date, fixing and calendar. | For a bill of face 1, discount quote d in decimals and D days, price=1−dD/360 and holding return=1/price−1. State interpolation/roll assumption or label annual-rate/day-count conversion as a proxy. Never use a US cash rate as CHF cash by default. |
| H05 · WRDS CRSP, Compustat and historical CCM | Security-month PERMNO/PERMCO, returns, delisting, price, shares, venue/share code; GVKEY, fiscal end, filing availability, book equity inputs, revenue, COGS, total assets; link dates/types and point-in-time industry. | Define exchange breakpoints, share classes, market cap units, accounting lag/restatements and delist conventions per dataset release. Keep firm-months together. Verify link uniqueness and report attrition across filters. |
| H05 · academic factor library | Market excess, SMB, HML, profitability, investment, momentum and cash as needed, monthly region/currency and units. H04 valuation data require dated numerator and denominator. | Align factors and forward return months explicitly. Convert percentage files to decimals once. Store model (CAPM/FF3/FF5 + momentum), input vintage and common sample. |
| H06 · authorized course StockTwits archive | Message or aggregate grain, timestamp/zone, asset identifier, polarity, count, event definition; market returns and dated company/sector links. | Drop duplicate IDs with an audit trail. Retain neutral and no-message days separately. Estimate pre-event market model, log coverage, avoid labels determined after trade time. |
| H03 exploratory materials and policy | Gold, oil or copper spot/total-return distinction, real yields and broad dollar; optional FOMC text with release timestamp/version. | Spot series can be predictors. A tradable commodity sleeve needs an investable returns series and documented roll/carry; separate explanation, forecast and execution. |
| H07 / monitoring · transaction inputs | Instrument/date: bid/ask or spread proxy, commissions, FX fee, settlement, liquidity flags, closing calendar and quote age. | If a spread is modeled, label scenarios and estimate break-even. Keep annual fund charges already embedded in fund returns from being charged twice. |
| Questionnaire · synthetic fixtures | Generated route, card version, choice, branch, pass, correction and holdout. | Keep held-out answers outside scoring. Interpret counts under the specified generator only. |


#### 3.1.1 · For each study separate tradable holdings, predictors, cash and benchmarks

The [series catalogue](research/source_catalogue/series_catalogue_v1.csv) has 115 study-role rows. Of these, 95 point to observed sources, seven specify benchmark rules and 13 record unmet study requirements. See the [seven study decisions](research/source_catalogue/README.md) for required versus available fields. The first observation date describes coverage. An executable or historically knowable start needs its own evidence.

**Universe sensitivity.** Using only pairs of consecutive ETF sessions with CHF quotes on both days, the three registered holdings and the full 13-instrument basket have different measured daily risks. This is a descriptive check of universe selection. The registered comparison retains the three original holdings.

#### 3.1.2 · Inventory authorized price/FX series, FRED/ALFRED macro vintages, WRDS security/accounting links, course StockTwits events and prospective spread/fee inputs

The [raw manifest](research/source_catalogue/raw_manifest_v1.csv) records 843 paths, 841 distinct hashes and 5,939,583,561 bytes. Every path has a count with a stated basis. Four WRDS exports have 54,356,930 daily CRSP, 2,639,046 monthly CRSP, 401,504 Compustat and 32,824 CCM records. Original retrieval timestamps survive for 46 paths. The ALFRED vintage date is available for all 786 snapshots. The full reload reproduced the manifest. Rebuild with the commands in the [inventory notes](research/source_catalogue/README.md#reproduction).

**Backfill check.** For each archived vintage use its last available observation. Compare the feature computed in that snapshot with the same observation from the final snapshot. CPI and industrial production use year-on-year growth within each vintage, so a mere change of index scale cancels. Unemployment uses the published level. Also count current-month observations present in the final file but absent from that month’s snapshot. A vintage is usable at the next eligible session under the registered rule. A change in the last-available feature is a revision diagnostic, not a return forecast.

In [ ]:
import csv
import gzip
import math
from pathlib import Path
from statistics import stdev

def coverage_sensitivity(data):
    with (data / "raw/etf_prices_adjusted.csv").open() as stream:
        prices = list(csv.DictReader(stream))
    with (data / "raw/public_phase2/DEXSZUS.csv").open() as stream:
        fx = {r["observation_date"]: float(r["DEXSZUS"]) for r in csv.DictReader(stream) if r["DEXSZUS"] not in ("", ".")}
    all_tickers = [s for s in prices[0] if s != "date"]
    returns = [[], []]
    for before, after in zip(prices, prices[1:]):
        day, prior = after["date"], before["date"]
        if not ("2020-01-01" <= day <= "2026-10-31" and day in fx and prior in fx):
            continue
        for values, tickers in zip(returns, (["SPY", "IEF", "GLD"], all_tickers)):
            gross = sum(float(after[s]) / float(before[s]) for s in tickers) / len(tickers)
            values.append(gross * fx[day] / fx[prior] - 1)
    print(f"Common CHF daily returns: {len(returns[0])}; daily SD, 3 vs 13 instruments: "
          f"{stdev(returns[0]):.3%} / {stdev(returns[1]):.3%}")

def vintage_values(path):
    with gzip.open(path, "rt", newline="") as stream:
        rows = csv.reader(stream)
        next(rows)
        return {day: float(v) for day, v in rows if v not in ("", ".")}

def revision_check(data):
    folder = data / "raw/public_phase2/alfred"
    for series in ("CPIAUCSL", "INDPRO", "UNRATE"):
        files = sorted(folder.glob(series + "_*.csv.gz"))
        latest = vintage_values(files[-1])
        compared = revised = unavailable = 0
        for path in files:
            then = vintage_values(path)
            origin = path.name.split("_")[1][:10]
            # Backdating a later snapshot to the observation month can invent availability.
            month = origin[:7] + "-01"
            unavailable += month in latest and month not in then
            day = max(then)
            previous = str(int(day[:4]) - 1) + day[4:]
            if day not in latest or (series != "UNRATE" and (previous not in latest or previous not in then)):
                continue
            def feature(values):
                return values[day] if series == "UNRATE" else 100 * (values[day] / values[previous] - 1)
            compared += 1
            revised += not math.isclose(feature(then), feature(latest), abs_tol=1e-8, rel_tol=0)
        print(f"{series}: {len(files)} vintages; {unavailable} later-known current-month slots; "
              f"{revised}/{compared} last-available features differ from final snapshot.")
data = Path("data")
if (data / "raw/etf_prices_adjusted.csv").is_file() and (data / "raw/public_phase2/DEXSZUS.csv").is_file():
    coverage_sensitivity(data)
else:
    print("Local price/FX history unavailable; see the recorded source audit.")
if len(list((data / "raw/public_phase2/alfred").glob("*.csv.gz"))) == 786:
    revision_check(data)
else:
    print("Complete vintage set unavailable; see the recorded revision check.")

Common CHF daily returns: 1683; daily SD, 3 vs 13 instruments: 0.671% / 0.994%
CPIAUCSL: 262 vintages; 260 later-known current-month slots; 253/262 last-available features differ from final snapshot.
INDPRO: 262 vintages; 261 later-known current-month slots; 261/262 last-available features differ from final snapshot.
UNRATE: 262 vintages; 260 later-known current-month slots; 117/262 last-available features differ from final snapshot.


#### 3.1.3 · Build fictional shareable fixtures alongside restricted local data

The [fictional ledger](data/demo/source_events_v1.json) stores raw events and manually computed expected rows. The standard-library checker recomputes units, settled CHF, wealth and the macro value known before each date. Expected wealth in CHF is 240, 240, unknown, 233.60, 229 and 219. A CHF 61 withdrawal against CHF 60 settled cash must fail. The checker runs from a clean, sparse checkout with only the script and fixture.

Reproduce with `python -I -S research/check_source_fixture.py`. The synthetic CI job exercises this path.

#### 3.1.4 · Verify the local data import

The linked [import report](data/README_data.md) records the full read-only check, including WRDS and ALFRED fingerprints and raw-to-prepared return reconciliation. The following independent, optional smoke check reads only the benchmark CSV. A clone without local data runs the synthetic cells. The 13 series define this imported panel only. Hash agreement identifies a vintage; it does not establish historical availability.

**Bias and control.** An imported table can reproduce an earlier error. Raw-to-prepared ETF returns were independently reconciled; the provenance, publication timing and construction of the WRDS panel require separate tests.

In [ ]:
import csv
import hashlib
import json
from pathlib import Path
from math import isfinite

price_path = Path("data/raw/etf_prices_adjusted.csv")
if price_path.is_file():
    with price_path.open("rb") as stream:
        observed_hash = hashlib.file_digest(stream, "sha256").hexdigest()
    expected = json.loads(Path("data/pilot_checksums.json").read_text())
    assert observed_hash == expected[price_path.as_posix()]
    with price_path.open(newline="") as stream:
        reader = csv.DictReader(stream)
        assets = [name for name in reader.fieldnames if name != "date"]
        dates = []
        for row in reader:
            dates.append(row["date"])
            assert all(isfinite(float(row[name])) and float(row[name]) > 0 for name in assets)
    assert dates and dates == sorted(set(dates))
    print(len(dates), "sessions;", len(assets), "series;", dates[0], "to", dates[-1])
else:
    print("Local benchmark CSV unavailable. Synthetic walkthrough remains runnable.")

### 3.2 Clean without leaking tomorrow's information

**Evidence to retain:** prepared panel, join attrition and exceptions. **Course reading:** [Guide Ultime Chapitre 01](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_01.md) · [guide tris portefeuille asset growth](Theory/Course_Notes/guide_tris_portefeuille_asset_growth.md).

#### 3.2.1 · Normalize calendars, time zones, quote direction and identifiers

**Run and evidence.** The venue/session check produced 71,565 rows and zero duplicate market keys. The price snapshot preceded the last official close, so 13 provisional marks are retained as raw and excluded from returns. Of 71,539 USD returns, 70,161 have the required adjacent FX quotes. On common dates the SPY daily CHF standard deviation is 1.318% with same-session FX, 1.331% with a one-session lag and 1.282% with a lead. The 29 MAD flags are diagnostic. XLF and XLE each have one unresolved distribution-date source difference above 10 bp. [Market exceptions](data/prepared_v1/market_exceptions.csv) · [Asset/year audit](data/prepared_v1/quality_by_year.csv).

#### 3.2.2 · Separate observation, publication, availability and execution

**Run and evidence.** The release stores 786 ALFRED snapshots and reconstructs the latest unemployment level or twelve-month growth from within each vintage. Decisions use the UTC exchange close and the latest release made available strictly beforehand. Day-only releases use the next eligible open. The mutation checks alter a future vintage, an after-close release and the version of an earlier share price. Earlier signals, example weights, filters and unit counts stay identical. Exact publication and broker execution times are unknown in these sources. Daily spread/FX histories carry retrospective status. [Timing and tests](research/cleaning/README.md#which-information-was-knowable).

In [ ]:
# Release-day ordering. Each row is (publication time, observation month, value).
releases = [("2025-02-12T13:30", "Jan", 2.0), ("2025-03-12T13:30", "Jan", 2.2)]
def as_of(rows, cutoff):
    known = [row for row in rows if row[0] < cutoff]
    return max(known)[2] if known else None
cutoff = "2025-02-12T16:00"
assert as_of(releases, cutoff) == as_of(releases[:1], cutoff) == 2.0
assert as_of(releases, "2025-02-12T12:59") is None

#### 3.2.3 · Reconcile splits, dividends, delisting returns, security entry/exit and cash-rate conventions

**Run and evidence.** XLK’s two-for-one split on 5 December 2025 yields a 0.732% return from already split-normalized closes. SPY’s 18 March 2005 distribution gives approximately −0.296% from the closing prices plus USD 0.467 cash. For PERMNO 11300, five daily returns compound to −6.63554% against the January 1998 CIZ monthly total of −6.6355%. All 16,683 terminal candidates have a monthly return. On 5,367 uncertain exits, adverse proceeds assumptions are retained separately. Across the candidate terminal rows, mean return is −7.61% as reported, −14.63% with the additional 30% stress and −31.00% with the total-loss stress. These are exit-row averages. The 91-day discount-bill proxy differs from simple accrual by at most 0.0443 bp per interval. [Reconciliations and sources](research/cleaning/README.md#corporate-actions-cash-and-exits).

In [ ]:
# Synthetic corporate actions and quoted FX conventions.
assert isclose((50 * 2) / 100 - 1, 0)  # two-for-one split
assert isclose((99 + 1) / 100 - 1, 0)  # cash distribution
assert isclose((1.1 * 0.5) - 1, -0.45)  # separate regular and delisting gross returns

#### 3.2.4 · Verify publication timing of fundamentals, macro revisions and sector classifications

**Run and evidence.** The raw CRSP projection contains 2,607,985 distinct security-months. Dated CCM links are ranked by primacy and link type. Equal-priority distinct companies would be excluded. Of 30,082 tagged firm-days, 3,676 carry a unique effective sector, 3,836 lack a same-day PERMNO and 22,570 lack an effective sector after matching. Zero have verified publication dates. The imported accounting panel has 48,506 assumed availability dates after the trading session, including 11,983 formerly marked investable. The H05 predictive sample is empty under the verified-filing rule. [Subgroup coverage](data/prepared_v1/sector_coverage.csv) · [Security-month coverage](data/prepared_v1/company_coverage.csv).

#### 3.2.5 · Freeze common trading dates, warm-up, eligible assets and missing-data rules per comparison

**Run and evidence.** The [monthly eligibility table](data/prepared_v1/sample_coverage.csv) retains each method and original calendar, including fallback and incomplete outcomes. In historical replay H01 and H07 have 65/82 fully covered months. EWMA versus rolling has 53/82 and GARCH has 0/82 under the contiguous-history rule. H03 and H04 have 82/82 retrospective input comparisons, H05 has 0/84, and H06 has 20/26. These are data-eligibility counts, not performance estimates. A missing quote defers the order. Actual executions remain unknown. Rebuild all tables with `python research/prepare_panels.py` and run `python tests/test_cleaning.py` using requirements-data.txt. [Sample rules](research/cleaning/README.md#monthly-eligibility-and-interpretation).

In [ ]:
# Audit the stored release with the standard library, including its failure denominators.
import csv
import hashlib
import json
from pathlib import Path

release = Path("data/prepared_v1")
if release.is_dir():
    hashes = json.loads((release / "manifest.json").read_text())
    for name, expected in hashes.items():
        with (release / name).open("rb") as stream:
            assert hashlib.file_digest(stream, "sha256").hexdigest() == expected, name
    with (release / "sample_coverage.csv").open() as stream:
        for row in csv.DictReader(stream):
            if row["phase"] == "historical replay":
                print(row["study"], row["arm"], "· eligible versus reference with complete outcomes:",
                      row["reference_pairs"], "/", row["origins"])
    print(len(hashes), "verified release files")
else:
    print("Prepared historical release unavailable in this checkout. See the stored 3.2 evidence.")

H01 Ledoit-Wolf · eligible versus reference with complete outcomes: 65 / 82
H01 sample covariance · eligible versus reference with complete outcomes: 65 / 82
H02 EWMA · eligible versus reference with complete outcomes: 53 / 82
H02 GARCH-t · eligible versus reference with complete outcomes: 0 / 82
H02 rolling 63 · eligible versus reference with complete outcomes: 53 / 82
H03 slope + macro PC · eligible versus reference with complete outcomes: 82 / 82
H03 slope OLS · eligible versus reference with complete outcomes: 82 / 82
H04 OLS · eligible versus reference with complete outcomes: 82 / 82
H04 expanding mean · eligible versus reference with complete outcomes: 82 / 82
H04 ridge · eligible versus reference with complete outcomes: 82 / 82
H05 boosting · eligible versus reference with complete outcomes: 0 / 84
H05 elastic net · eligible versus reference with complete outcomes: 0 / 84
H05 four-characteristic rank · eligible versus reference with complete outcomes: 0 / 84
H06 no-view anchor ·

## 4. Describe historical returns and macro exposures

In [ ]:
path = [100, 120, 90, 110]
drawdowns = [value / max(path[:i + 1]) - 1 for i, value in enumerate(path)]
assert isclose(min(drawdowns), -0.25)

### 4.1 Produce descriptive evidence

**Evidence to retain:** historical metrics and reconciliation. **Course reading:** [EMiF Class2 Version Ultra Pedagogique fixed](Theory/Course_Notes/EMiF_Class2_Version_Ultra_Pedagogique_fixed.md) · [Guide Ultime Chapitre 05](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_05.md).

#### 4.1.1 · Calculate simple, log, excess and compounded returns with explicit calendar and currency

Calculate simple, log, excess and compounded returns with explicit calendar and currency. Compare manual one-period and multi-period examples to code. Plot wealth and asset coverage together.

**Procedure.** Use r=P_TR,t/P_TR,t−1−1, log1p(r), cumulative wealth Π(1+r), excess r−r_f. Document return currency, timestamps and cash convention. Compute annual growth from actual elapsed years.

**Verification and failure handling.** Hand-check a two-period +10%, −10% path ends at 0.99 of initial capital. Report calendar gaps, missing and total-loss observations separately.

**Bias and control.** Arithmetic/geometric confusion. Show arithmetic mean and realized CAGR with units, fees and exact sample dates; use compounded cash flows for wealth.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

#### 4.1.2 · Compute annualized volatility, drawdown depth/duration, rolling correlations, historical VaR and Expected Shortfall

Compute annualized volatility, drawdown depth/duration, rolling correlations, historical VaR and Expected Shortfall. Record tail counts, ties and interval uncertainty. Test a flat path and a known loss path.

**Procedure.** Using positive losses, estimate VaR as the chosen empirical quantile and ES as the average worst tail mass with fractional boundary weight. Plot running drawdown W/max(W)−1 and recovery duration. For annual volatility disclose serial-dependence assumptions.

**Verification and failure handling.** Backtest time-ordered VaR exceptions, exact binomial counts under independence and clustering diagnostics. Report uncertainty with blocks, insufficient tail size and ongoing unrecovered drawdowns.

**Bias and control.** Sparse and dependent tails. Report quantile convention, number of independent episodes, breach clustering and interval uncertainty. For joint VaR/ES comparisons preregister a consistent joint score with matching return/loss sign and domain.

**Tail calibration and annualization.** At frozen one-step VaR origins report predicted versus actual breach counts. Use Kupiec unconditional coverage and Christoffersen independence/conditional coverage when exception and transition counts support inference. Otherwise show counts, sparse transitions and dependence-aware uncertainty. For covariance-stationary additive returns use Var(sum of h returns)=hγ₀+2Σ(h−k)γₖ for k=1…h−1. Compare the IID square-root-of-time shortcut with autocovariance sensitivity; compounded simple returns require pathwise compounding.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

In [ ]:
# Synthetic additive returns. An AR(1)-style autocovariance alters horizon risk.
def horizon_variance(gamma, horizon):
    if horizon < 1 or len(gamma) < horizon:
        raise ValueError("Supply autocovariances from lag zero to horizon minus one.")
    return horizon * gamma[0] + 2 * sum((horizon-k)*gamma[k] for k in range(1, horizon))
assert isclose(horizon_variance([.01, 0, 0], 3), .03)
assert isclose(horizon_variance([.01, .005, .0025], 3), .055)
# These are variances of a sum. Compound individual returns for a wealth path.

In [ ]:
# Positive losses, fractional empirical ES with 1.5 observations in the tail.
from math import floor
losses = sorted([0.01, -0.01, 0.02, 0.08, 0.06, 0.00], reverse=True)
mass = 0.25 * len(losses)
full = floor(mass)
es = (sum(losses[:full]) + (mass-full)*losses[full]) / mass
assert isclose(es, (0.08 + 0.5*0.06) / 1.5)

#### 4.1.3 · Attribute portfolio variance and marginal risk to holdings

Attribute portfolio variance and marginal risk to holdings. Check covariance symmetry, eigenvalues and conditioning. Verify contributions sum to total risk and examine sensitivity to the estimation window.

**Procedure.** Compute σp=√(w′Σw), RCᵢ=wᵢ(Σw)ᵢ/σp and effective holdings 1/Σwᵢ² for fully invested long-only positions. Inspect correlation heatmaps, eigenvalues and condition number.

**Verification and failure handling.** Check ΣRCᵢ=σp; identical assets give diversification ratio one. Diagnose singularity before solving. Preserve negative hedging contributions.

**Bias and control.** Noise in diversification estimates. Cross-check risk contributions across windows and joint resamples. When volatility is near zero or covariance indefinite, flag attribution as undefined or unreliable.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

In [ ]:
from math import sqrt
w = [0.5,0.5]
cov = [[0.04,0.01],[0.01,0.09]]
cw = [sum(x*y for x,y in zip(row,w)) for row in cov]
sigma = sqrt(sum(x*y for x,y in zip(w,cw)))
contributions = [x*y/sigma for x,y in zip(w,cw)]
assert isclose(sum(contributions),sigma)

### 4.2 Explain reactions to shocks

**Evidence to retain:** dated exposures and stress attribution. **Course reading:** [W3 Economic Analysis Guide FR](Theory/Course_Notes/W3_Economic_Analysis_Guide_FR.md) · [Guide Ultime Chapitre 12](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_12.md).

#### 4.2.1 · Specify common rate, equity, inflation and FX shocks

Specify common rate, equity, inflation and FX shocks. Estimate sensitivities using dated exposures with assumptions stated. Apply identical shocks across allocations and reconcile holding contributions to the portfolio effect.

**Procedure.** For fixed income use −modified duration × Δyield plus ½ convexity × Δyield² when inputs exist. For other assets estimate dated factor exposures and apply an explicit vector of market, FX or inflation shocks.

**Verification and failure handling.** Sum holding-level P&L to portfolio P&L. Test bp-to-decimal conversion and cash sensitivity. Distinguish residual risk from explained shock loss.

**Bias and control.** Double-counted macro channels. Separate direct FX repricing from local-asset exposures. Validate combined scenario versus a joint revaluation, and report interaction residuals.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

#### 4.2.2 · Compare calm and stressed samples, historical episodes and hypothetical scenarios

Compare calm and stressed samples, historical episodes and hypothetical scenarios. Report observation counts, exposure instability and confounding. Label any association requiring separate predictive or causal evidence.

**Procedure.** Define crisis windows and identification assumptions before comparing outcomes. Estimate rolling exposures with uncertainty. For regime models use filtered probabilities available at t.

**Verification and failure handling.** Compare common dates and counts. Run leave-one-crisis-out sensitivity. A shock association alone cannot support a causal or predictive claim.

**Bias and control.** Ex-post regime selection. Predefine episodes and run leave-one-episode-out comparisons. For a regime strategy archive filtered probabilities and compare with an unconditional policy.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

## 5. Replicate research and validate forecasts

In [ ]:
# Illustrate interval evaluation on invented chronological outcomes.
intervals = [(0, 2), (0, 1), (-1, 1)]
outcomes = [1, 2, 0]
coverage = sum(lo <= y <= hi for (lo, hi), y in zip(intervals, outcomes)) / len(outcomes)
assert isclose(coverage, 2 / 3)
print("Toy coverage:", coverage, "Observations:", len(outcomes))

### 5.1 Run each registered experiment

**Evidence to retain:** paired origin-level predictions, holdings and diagnostic ledger. **Course reading:** [Guide Ultime Chapitre 01](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_01.md) · [literature](Theory/Course_Notes/literature.md).

#### 5.1.1 · Compare sample and Ledoit–Wolf covariance at identical weights using NumPy/scikit-learn

Compare sample and Ledoit–Wolf covariance at identical weights using NumPy/scikit-learn. Score future integrated variance (QLIKE), check conditioning, then rerun optimization and measure weight stability and net utility.

**Procedure.** Fit sample covariance and sklearn LedoitWolf to the same trailing past returns. At frozen weights aggregate predicted daily variance to next month's integrated variance. Realized proxy Σdaily(w′r_d)² with a declared centering convention. Compare QLIKE(v,y)=log(v)+y/v.

**Verification and failure handling.** Require positive v and identical units/dates. Report paired loss difference, matrix conditioning, weight dispersion and net utility after re-optimization. Attribute forecast and allocation effects separately.

**Study sources.** [QARM2 LN3 Cours Explique copie 2](Theory/Quant_Project/QARM2_LN3_Cours_Explique%20copie%202.md) · [EMiF Class11 Version Ultra Pedagogique fixed](Theory/Course_Notes/EMiF_Class11_Version_Ultra_Pedagogique_fixed.md).

**Bias and control.** Target mismatch. Summed daily conditional variance measures integrated variance, while variance of a cumulative return may include autocovariances. Freeze the target and weights for the entire measurement. Check the sklearn shrinkage target against the replicated paper specification.

**Dynamic dependence as a conditional challenger.** Use DCC-GARCH only after checking the individual variance fits, standardized residuals, convergence and positive-definite H_t=D_t R_t D_t. Compare rolling, shrinkage and dynamic covariance at the same information cutoff, fixed weights and evaluation dates. Record fallback performance and whether the extra model changes after-cost allocations.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

In [ ]:
from math import log
def qlike(variance, realized):
    assert variance > 0 and realized >= 0
    return log(variance) + realized / variance
assert qlike(0.04, 0.04) < qlike(0.08, 0.04)

#### 5.1.2 · Compare rolling variance, EWMA and GARCH-t with GJR sensitivity using arch

Compare rolling variance, EWMA and GARCH-t with GJR sensitivity using arch. Check convergence, persistence, Ljung–Box and ARCH-LM on standardized residuals. Match forecast and realized variance horizons. Test dynamic exposure separately after costs.

**Procedure.** Fit rolling baseline, EWMA and arch GARCH(1,1)-t: h_t=ω+αε²_(t−1)+βh_(t−1). Add GJR asymmetry as a registered variant. Refit only at admissible origins. Sum daily conditional variance forecasts over common horizon.

**Verification and failure handling.** Check positivity, fit convergence, finite innovation variance and persistence. Inspect standardized residuals and their squares via Ljung–Box, then ARCH-LM. Compare QLIKE and block intervals. Fall back to EWMA when fit fails; report failure frequency.

**Study sources.** [EMiF Classe9 ARCH GARCH fixed](Theory/Course_Notes/EMiF_Classe9_ARCH_GARCH_fixed.md) · [EMiF Classe10 GARCH fixed](Theory/Course_Notes/EMiF_Classe10_GARCH_fixed.md).

**Bias and control.** Failed-fit selection. Keep every forecast origin, optimizer warning and benchmark fallback. Score both paired successful fits and the full deployed rule. For Student-t check degrees of freedom >2 before variance interpretations.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

In [ ]:
# A one-step GARCH variance uses the previous shock, in decimal returns.
omega, alpha, beta = 0.00001, 0.1, 0.8
def next_variance(last_shock, last_variance):
    return omega + alpha*last_shock**2 + beta*last_variance
assert next_variance(-0.03, 0.0001) > next_variance(0.01, 0.0001)

#### 5.1.3 · Compare a yield-curve bond forecast with lagged macro factors learned by PCA inside training

Compare a yield-curve bond forecast with lagged macro factors learned by PCA inside training. For VAR or local projections justify lags, stability, shock identification and interval construction. Treat gold/rates and resources/sectors as exploratory until separately confirmed.

**Procedure.** Forecast next-month excess bond return using released curve slope as baseline. Within training scale and fit macro PCA, then augment baseline. For VAR use information criteria inside training, check roots, residual autocorrelation and ordering sensitivity. For local projections define shock, controls and horizon-specific HAC.

**Verification and failure handling.** Compare paired squared errors and out-of-sample R² on vintage-consistent dates. For impulse responses report identification assumptions and bootstrap bands. Treat exploratory real-rate/gold and resource/sector tests as separate hypotheses.

**Study sources.** [EMIF Class5-6 Version Ultra Pedagogique fixed](Theory/Course_Notes/EMIF_Class5-6_Version_Ultra_Pedagogique_fixed.md) · [W3 FED Cours Complet](Theory/Course_Notes/W3_FED_Cours_Complet.md).

**Bias and control.** Generated factors and identification. Refit scaling, PCA, lag selection and regression within every resampled training block. Apply alternative VAR orderings or external identification; mark responses sensitive to ordering as scenario evidence.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

In [ ]:
# A common +100bp parallel yield shock, first-order duration approximation.
duration = [0.0, 5.0]
weights_by_portfolio = [[0.8, 0.2], [0.2, 0.8]]
pnl = [-0.01 * sum(w*d for w, d in zip(weights, duration)) for weights in weights_by_portfolio]
assert all(isclose(x,y) for x,y in zip(pnl, [-0.01, -0.04]))

#### 5.1.4 · Compare expanding historical mean, OLS and ridge for forward equity excess return

Compare expanding historical mean, OLS and ridge for forward equity excess return. Tune on internal chronological folds. Report out-of-sample R² and paired losses. Apply Clark–West only for suitable nested squared-error forecasts.

**Procedure.** Refit expanding mean, lagged OLS and inner-tuned ridge at each origin. Compute R²_OS=1−Σe_model²/Σe_mean². For persistent predictors simulate the null with dependence retained; evaluate whether apparent fit is common under no predictability.

**Verification and failure handling.** Use Clark–West only for appropriate nested squared-error forecasts, paired dependence-aware tests for eligible alternatives. Report negative R², sensitivity to start date and after-cost effect.

**Study sources.** [guide predictibilite agregee marche](Theory/Course_Notes/guide_predictibilite_agregee_marche.md) · [Advanced Data Analytics Lecture4 Regression Explique](Theory/Course_Notes/Advanced_Data_Analytics_Lecture4_Regression_Explique.md).

**Bias and control.** Persistent predictors and small-sample optimism. Simulate joint predictor/return innovations under no predictability, preserving persistence and innovation correlation. Rerun the selection pipeline and compare observed forecast gain with its null distribution.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

In [ ]:
observed, benchmark, forecast = [0.02, -0.01], [0, 0], [0.01, 0]
def squared_error(actual, prediction):
    return sum((a-p)**2 for a,p in zip(actual, prediction, strict=True))
r2_os = 1 - squared_error(observed, forecast)/squared_error(observed, benchmark)
assert isclose(r2_os, 0.6)

**Working H04 baseline.** Map the dated monthly panel to the five fields below. Each origin must have a predictor available by the decision; an unresolved outcome is allowed for prediction. The training sample includes only labels strictly available before the current decision. Freeze the study sample, tuning and primary loss in the register before running this baseline on market data.

In [ ]:
# H04 reusable expanding-origin OLS versus historical mean.
# One row per origin. All timestamps must share one zone and be comparable.
from math import isfinite

def expanding_predictions(rows, minimum_train):
    if minimum_train < 3:
        raise ValueError("Set a minimum of at least three training observations.")
    ordered = sorted(rows, key=lambda row: row["origin"])
    if len({row["origin"] for row in ordered}) != len(ordered):
        raise ValueError("Duplicate forecast origins.")
    for row in ordered:
        known = row.get("label_known")
        actual = row.get("outcome")
        if (known is None) != (actual is None):
            raise ValueError("Record availability and realized outcome together.")
        if row["feature_known"] > row["origin"] or (known is not None and known <= row["origin"]):
            raise ValueError("Feature or label timing is inconsistent.")
        if not isfinite(row["predictor"]) or (actual is not None and not isfinite(actual)):
            raise ValueError("Apply and document the missing-data rule first.")
    predictions = []
    for row in ordered:
        train = [r for r in ordered if r.get("label_known") is not None and r["label_known"] < row["origin"]]
        if len(train) < minimum_train:
            continue
        mx = fmean(r["predictor"] for r in train)
        my = fmean(r["outcome"] for r in train)
        ssx = sum((r["predictor"] - mx)**2 for r in train)
        if ssx == 0:
            fitted, status = my, "constant predictor: mean fallback"
        else:
            slope = sum((r["predictor"]-mx)*(r["outcome"]-my) for r in train) / ssx
            fitted, status = my + slope*(row["predictor"]-mx), "OLS"
        predictions.append(dict(origin=row["origin"], n=len(train), benchmark=my,
                                forecast=fitted, status=status))
    return predictions

toy = [dict(origin=i*2, feature_known=i*2-1, label_known=i*2+1,
            predictor=i/100, outcome=(i+1)/100) for i in range(9)]
first = expanding_predictions(toy, 4)
changed = [dict(r, outcome=3) if r["origin"] >= 12 else r for r in toy]
second = expanding_predictions(changed, 4)
assert len(first) == 5
outcomes_by_origin = {r["origin"]: r["outcome"] for r in toy}
assert all(isclose(a["forecast"], outcomes_by_origin[a["origin"]]) for a in first)
assert [r["forecast"] for r in first if r["origin"] <= 12] == [
    r["forecast"] for r in second if r["origin"] <= 12]
print("Synthetic expanding-origin forecast rows:", len(first))
# The last origin has no known outcome. Its archived prediction must be identical.
live = toy[:-1] + [dict(toy[-1], outcome=None, label_known=None)]
assert expanding_predictions(live, 4) == first
assert all("actual" not in row for row in first)

#### 5.1.5 · Use DuckDB/WRDS to construct lagged value, profitability, asset growth and 12–1 momentum

Use DuckDB/WRDS to construct lagged value, profitability, asset growth and 12–1 momentum. Compare dated sorts, elastic net and boosting on whole-month blocks. Test Spearman rank correlation, factor alpha, microcap sensitivity and the net investable long-only sleeve.

**Procedure.** Within month form dated book-to-market, gross profitability, asset growth and 12–1 momentum. Declare accounting definitions, NYSE breakpoints, share-class and microcap rules. Compare simple sort, elastic net and boosting with whole months held together.

**Verification and failure handling.** Report Spearman rank IC per month, equal/value-weighted quantile spreads, long-only return and factor alpha with HAC. Rerun ex-microcaps, with estimated spread costs and dated industry controls. A diagnostic short leg needs separate borrow assumptions to be investable.

**Study sources.** [guide tris portefeuille asset growth](Theory/Course_Notes/guide_tris_portefeuille_asset_growth.md) · [guide modeles factoriels capm fama french](Theory/Course_Notes/guide_modeles_factoriels_capm_fama_french.md).

**Bias and control.** Errors in estimated betas and false factor discovery. If two-pass premia use estimated exposures, bootstrap both passes by month or justify a Shanken correction under its assumptions. For joint alpha testing check GRS sample size, nonsingularity and assumptions; otherwise use dependence-aware joint inference.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

In [ ]:
# Whole months remain together in a cross-sectional fold.
firm_months = [(firm, month) for month in (1, 2, 3) for firm in ("a","b","c")]
train_months = {m for _,m in firm_months if m < 3}
test_months = {m for _,m in firm_months if m == 3}
assert train_months.isdisjoint(test_months)

#### 5.1.6 · Replicate the LN7 pre-event market model, volume peaks, cumulative abnormal returns and polarity

Replicate the LN7 pre-event market model, volume peaks, cumulative abnormal returns and polarity. Control for overlapping events and available timestamps. Compare Black–Litterman with and without a view at matched tracking error, including no-signal dates.

**Procedure.** Estimate pre-event market model outside the event window; AR_it=r_it−(α_i+β_i r_mt), CAR=sum AR. Align message polarity, volume peaks and first executable price; distinguish no messages from neutral polarity. Replicate LN7 rank comparison then account for coincident dates.

**Verification and failure handling.** Freeze P (view exposures), Q (expected excess view return), Ω (uncertainty) and prior before execution. With Ω large recover the no-view solution. Compare net utility under the same risk budget including no-signal months. Unknown publication dates block the predictive branch.

**Study sources.** [QARM2 LN4 Cours Explique copie 2](Theory/Quant_Project/QARM2_LN4_Cours_Explique%20copie%202.md) · [QARM2 LN7 Cours Explique copie 2](Theory/Quant_Project/QARM2_LN7_Cours_Explique%20copie%202.md).

**Bias and control.** Event-conditioned hindsight. Build placebo events matched on calendar, liquidity and prior message volume, preserving clusters. Compare event-window results with pre-event diagnostics; exclude labels assigned after decision time from the signal.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

In [ ]:
# Market model coefficients are assumed fixed from a prior estimation window.
market, stock = [0.01, -0.02], [0.025, -0.015]
abnormal = [r - (0.001 + 1.2*m) for r,m in zip(stock,market)]
assert isclose(sum(abnormal), 0.020)

#### 5.1.7 · Run identical signals through monthly, quarterly and no-trade-band policies on drifted holdings

Run identical signals through monthly, quarterly and no-trade-band policies on drifted holdings. Compare net wealth, turnover, risk drift, withdrawal coverage and break-even unit costs.

**Procedure.** Apply monthly, quarterly and band rules to the same signals and cash flows with drifted weights. Use currency amounts traded for costing. Find unit-cost break-even by re-running the funded ledger over a cost grid.

**Verification and failure handling.** Report paired net difference, risk-limit breaches, overdue withdrawals and hold dates. If no sign change occurs disclose no crossing in the tested grid.

**Study sources.** [guide couts transaction trading](Theory/Course_Notes/guide_couts_transaction_trading.md) · [Guide Ultime Chapitre 06](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_06.md).

**Bias and control.** Policy tuning bias. Select a band on inner validation only. Compare on common post-selection dates and report near-optimal bands, turnover and funding outcomes when the chosen threshold shifts.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

In [ ]:
# Half-spread and commission per traded dollar, from drifted weights.
current, target = [65,35], [60,40]
traded = sum(abs(a-b) for a,b in zip(current,target))
assert traded == 10 and isclose(traded * 0.001, 0.01)

### 5.2 Apply econometric and forecast controls

**Evidence to retain:** split manifest, residual and calibration report. **Course reading:** [EMiF Classe7 Version Ultra Pedagogique fixed](Theory/Course_Notes/EMiF_Classe7_Version_Ultra_Pedagogique_fixed.md) · [EMIF Class5-6 Version Ultra Pedagogique fixed](Theory/Course_Notes/EMIF_Class5-6_Version_Ultra_Pedagogique_fixed.md) · [Advanced Data Analytics Lecture11 DimReduction Explique](Theory/Course_Notes/Advanced_Data_Analytics_Lecture11_DimReduction_Explique.md) · [Guide Ultime Chapitre 17](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_17.md).

#### 5.2.1 · Inspect ACF/PACF, trends, outliers, dependence and breaks

Inspect ACF/PACF, trends, outliers, dependence and breaks. Select applicable ADF/KPSS specifications, HAC or clustered inference. For integrated systems test cointegration before VECM. For panels justify fixed effects or Fama–MacBeth and corresponding inference.

**Procedure.** Follow guide 01 from population and variable to specification, estimation and residuals. ADF null is unit root; KPSS null is stationarity. Specify constant/trend and lag choices. For panels consider firm/month effects and dependence along both dimensions. For factor premia run monthly cross-sections then time-series inference on premia.

**Verification and failure handling.** Conflicting stationarity results trigger break and transformation sensitivity. For integrated systems test Johansen rank with deterministic terms before VECM. Document sample loss, residual plots, serial dependence and heteroskedasticity before interpreting coefficients.

**Bias and control.** Misspecification and endogeneity. Review residual plots, influence by date/firm and plausible omitted channels. Test alternate trend/lag choices on development; use predictive language in the absence of identified exogenous variation.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

#### 5.2.2 · Fit scaling, imputation, PCA, feature selection and tuning inside each training window

Fit scaling, imputation, PCA, feature selection and tuning inside each training window. Purge overlapping labels where required, retain all firms from one month in the same fold, and perturb future observations to detect leakage.

**Procedure.** Create rolling origins and nested inner splits. Fit sklearn Pipeline imputation, StandardScaler, PCA and hyperparameter search using inner training only. Exclude training labels whose end overlaps validation/test. Use filtered regimes only.

**Verification and failure handling.** Check train/validation/test timestamp inequalities and rerun after changing held-out values. Hyperparameters, past transforms and decisions must be unaffected by later data.

**Bias and control.** Cross-fold contamination. Archive feature fit dates and each training label's availability, including publication gaps. Test prediction at the latest origin while its outcome is still unknown.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

#### 5.2.3 · Archive dated point/range forecasts by asset, horizon, currency and version

Archive dated point/range forecasts by asset, horizon, currency and version. Score means by squared error, variance by QLIKE, quantiles by pinball loss. Report interval width, coverage, clustered exceedances and counts by regime.

**Procedure.** Log forecast origin, target end, model version, point forecast, lower/upper quantiles and realization. Pinball loss=(q−1[y<f])(y−f); for a central band report mean width and fraction inside. Use PIT/CRPS only for complete predictive distributions.

**Verification and failure handling.** Report sample counts, misses by side, consecutive misses and calibration by regime. Compare widths at similar coverage. Low coverage triggers recalibration on development or a broader fallback, never tuning on final test outcomes.

**Bias and control.** Calibration hidden by pooling. Evaluate interval coverage, width and separate upper/lower misses within prespecified subperiods using dependence-aware intervals. Keep thin groups unresolved.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Horizon and fallback.** Record elicitation horizon, forecast horizon, decision date and cash-flow dates separately. Monthly signals support near-term review. Evaluate long-term funding with explicit paths, reinvestment, inflation, FX and withdrawals. Block direct repetition of a one-month premium over the goal horizon. When an overlay expires, fails timing checks or fails out-of-sample admission, use the frozen baseline or no-view mix, mark the reason and preserve the failed origin. A stale valuation feed suspends a trade proposal.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

In [ ]:
def pinball(y, forecast, q):
    return (q - int(y < forecast)) * (y - forecast)
assert isclose(pinball(0, 1, 0.9), 0.1)
assert isclose(pinball(1, 0, 0.9), 0.9)

#### 5.2.4 · Compare paired losses on identical dates with suitable long-run variance or block bootstrap

Compare paired losses on identical dates with suitable long-run variance or block bootstrap. Record all variants and multiplicity. Retain simple fallback forecasts where gains are weak, unstable or too small after costs.

**Procedure.** Choose primary paired difference and economic threshold in advance. Use HAC or moving-block bootstrap of dated loss differentials, with horizon, block choice and sensitivity recorded. For multiple confirmatory variants declare family and e.g. Holm adjustment; record all attempts.

**Verification and failure handling.** Report effect, interval, sample size, rejected/retained hypotheses and net utility. Inconclusive results or a failed timing check keep the reference strategy in place.

**Bias and control.** Winner's curse. When selecting the best among many trading rules, use a joint block-resampled max-statistic or suitable SPA/Reality Check against the common benchmark. Center under the null and resample identical dates jointly across alternatives. Retain the multiplicity register and independent confirmation.

**Overlapping forecast losses.** For suitable non-nested forecasts, use Diebold–Mariano on paired d_t=L_candidate−L_reference with HAC long-run variance. Document target horizon, overlap, lag bandwidth, refitting scheme and small-sample sensitivity. If the differential has zero long-run variance, too few independent origins, or nesting invalidates the reference distribution, report the paired effect and its limitations; use the registered nested-model procedure when justified.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Forecast selection and combination.** For forecasts of the same target, currency and horizon at the same decision time, preregister a baseline, standalone candidates and an equal-weight forecast combination. If justified, compare nonnegative weights summing to one learned only from past out-of-fold forecasts and already available outcomes. Freeze the missing-component rule, compare paired forward losses and coverage on common origins, then evaluate the deployed rule across all origins. Check error dependence and incremental value against the baseline and individual forecasts. Retain the baseline, one model or a combination according to the frozen decision rule. Use explanatory-only signals for stress attribution. Require a separate after-cost benefit before changing allocation.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

In [ ]:
# Joint circular-block maximum-mean test on aligned, same-unit loss improvements.
# Positive values favor a candidate over the fixed reference. Stationary paired
# differences and a finite prespecified family are assumptions of this illustration.
from random import Random
from statistics import fmean
def max_gain_test(paired_dates, block, draws, seed):
    count = len(paired_dates)
    if count < 2 or not 1 <= block < count or draws < 1:
        raise ValueError("Invalid sample, block length or number of draws.")
    width = len(paired_dates[0])
    if not width or any(len(row) != width for row in paired_dates):
        raise ValueError("Align all candidates on common dates first.")
    means = [fmean(col) for col in zip(*paired_dates)]
    observed = max([0, *means])
    rng = Random(seed)
    exceed = 0
    for _ in range(draws):
        sample = []
        while len(sample) < count:
            start = rng.randrange(count)
            sample.extend((start + j) % count for j in range(block))
        # One set of dates per draw preserves cross-candidate dependence.
        null_max = max([0] + [fmean(paired_dates[i][k] for i in sample[:count]) - means[k]
                              for k in range(width)])
        exceed += null_max >= observed
    return (1 + exceed) / (1 + draws)

paired = [(0.02, -0.02), (-0.01, 0.01), (0.02, -0.02), (0, 0)]
assert max_gain_test([(0,0)]*4, 2, 49, 7) == 1
assert max_gain_test(paired, 2, 49, 7) == max_gain_test([r[::-1] for r in paired], 2, 49, 7)
# Select block length, appropriate statistic and economic threshold before market evaluation.

In [ ]:
# Circular block bootstrap of paired losses, for algorithm verification only.
from random import Random
from statistics import mean
def paired_block_means(differences, length, draws, seed):
    rng = Random(seed)
    n = len(differences)
    assert 1 <= length <= n
    result = []
    for _ in range(draws):
        sample = []
        while len(sample) < n:
            start = rng.randrange(n)
            sample.extend(differences[(start+j) % n] for j in range(length))
        result.append(mean(sample[:n]))
    return result
assert paired_block_means([1]*8, 3, 25, 7) == [1]*25

## 6. Build feasible portfolios and match a mandate

In [ ]:
# Same annual horizon. Returns already net of assumed costs.
options = {"lower volatility": (0.03, 0.05), "higher volatility": (0.05, 0.15)}
for phi in (1, 4):
    utilities = {name: mu - phi * vol**2 / 2 for name, (mu, vol) in options.items()}
    print(phi, max(utilities, key=utilities.get), utilities)

### 6.1 Estimate the constrained net frontier

**Evidence to retain:** frontier grid, feasibility status and weight sensitivities. **Course reading:** [QARM2 LN1 Cours Explique copie 2](Theory/Quant_Project/QARM2_LN1_Cours_Explique%20copie%202.md) · [QARM2 LN5 Cours Explique copie 2](Theory/Quant_Project/QARM2_LN5_Cours_Explique%20copie%202.md) · [QARM2 LN6 Cours Explique copie 2](Theory/Quant_Project/QARM2_LN6_Cours_Explique%20copie%202.md).

#### 6.1.1 · Apply liquidity, reserves, ethical exclusions, concentration and currency restrictions before optimization

Apply liquidity, reserves, ethical exclusions, concentration and currency restrictions before optimization. Define feasible cash and risky sleeves. Test empty eligible universe, all-cash and infeasible-risk cases.

**Procedure.** Create dated eligibility matrix by mandate and instrument (liquidity, exclusions, currency, limits, withdrawal funding). Run a feasibility problem before maximizing return.

**Verification and failure handling.** Report binding constraints and uncovered amounts. An empty sleeve or impossible target yields an explicit status; never relax a hard exclusion silently.

**Bias and control.** Dated eligibility and infeasible reserves. Verify exclusions and look-through exposures as of each decision. Unknown holdings or unsettled proceeds may invalidate an option; show an unresolved constraint explicitly.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

#### 6.1.2 · Compare equal weight, fixed mix, constrained minimum variance, risk parity and conditional strategies on identical inputs

Compare equal weight, fixed mix, constrained minimum variance, risk parity and conditional strategies on identical inputs. Check equal risk contributions in a known diagonal case and recovery of the Black–Litterman prior with negligible view confidence.

**Procedure.** Use equal weight and fixed mix controls, GMV and ERC (RCᵢ/σp=bᵢ); conditional BL prior π=δΣw_market in excess-return units. For LN2 compute beta=cov(p,m)/var(m), tracking error=sd(r_p−r_b).

**Verification and failure handling.** Analytically verify inverse-vol ERC under diagonal Σ, zero tracking error versus self, no-view BL and diversification ratio. Report benchmarks that lack a defensible prior.

**Bias and control.** Benchmark shopping. Freeze admissible controls before optimization. For an expanded universe test incremental opportunity by constrained out-of-sample spanning or matched-frontier comparison at common risk budgets after costs.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Fair comparator.** Before looking at returns define an implementable fixed mix and a rule for ineligible holdings or a binding risk cap. Freeze the mandate including reserve, currency, exclusions, fees, execution, cash flows and start holdings across arms. If a fixed mix violates a hard cap, apply its prespecified risk reduction or flag it infeasible. Rerun mismatched comparisons before interpreting gains.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

#### 6.1.3 · Sweep risk budgets with CVXPY or SciPy

Sweep risk budgets with CVXPY or SciPy. Estimate net expected return under explicit trade, spread and FX conventions. Save solver status, residuals, binding constraints and dominated candidates. Validate on a hand-computable two-asset case.

**Procedure.** For each cap s solve max μ′w−cost(Δw) subject to w′Σw≤s², budget and mandate constraints. Define financing of first trade and horizon of recurring costs. Use CVXPY and record primal residuals, feasibility and solver tolerances.

**Verification and failure handling.** Compare with analytic two-asset GMV and direct grid enumeration. Reject materially infeasible weights and dominated points on common net/currency assumptions. Label solver failures separately from economically infeasible caps.

**Bias and control.** Hidden financing in proportional fees. Express trades and budget in common pre-trade wealth units; positions plus reserve/cash plus one-time costs must equal wealth. Validate the initial purchase and the drifted rebalance independently.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Dual verification (guide 00, §0D.3.4).** For a convex feasible instance record the cap convention (volatility or variance), units, primal residual, dual multiplier and solver status. At a smooth active cap perturb the same right-hand side in both directions and compare the objective slope with the reported multiplier. Repeat with a smaller step and an inactive cap. At a kink use one-sided slopes. This is local sensitivity to a constraint, not a forecast.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

In [ ]:
# Charge cost on purchased notional and conserve pre-trade wealth.
wealth, fee_rate = 100.0, 0.01
purchased = wealth/(1+fee_rate)
fee = purchased*fee_rate
assert isclose(purchased+fee,wealth)
assert purchased < wealth

In [ ]:
# Fully invested two-asset GMV, diagonal covariance, no fees.
v1, v2 = 0.04, 0.01
w1 = v2 / (v1 + v2)
def portfolio_variance(weight):
    return weight**2*v1 + (1-weight)**2*v2
assert isclose(w1, 0.2)
assert isclose(portfolio_variance(w1), min(portfolio_variance(i/100) for i in range(101)))

#### 6.1.4 · Perturb means, covariances, fees and constraints and resample blocks

Perturb means, covariances, fees and constraints and resample blocks. Report weight and frontier dispersion. Flag rankings that flip under plausible inputs.

**Procedure.** Use moving blocks of return vectors shared across all assets, then re-estimate means/covariance and reoptimize. Compare sampling sensitivity with deterministic fee and cap scenarios.

**Verification and failure handling.** Report weight intervals, turnover, rank changes and constraint breaches including failed draws. An unstable frontier is shown as uncertain.

**Bias and control.** Conditional-on-success frontiers. Publish number of infeasible/failed draws at each risk cap and denominator of reported percentiles. Compare changing weights and changing feasible set separately.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Classical versus robust frontier (guide 04, §4.5.9).** At common risk budgets compare fixed mix, classical max net mean and robust max net mean minus ρ√(w′Vμw). Use Σ for risk and Vμ for uncertainty of the estimated horizon mean. Estimate Vμ from mean vectors obtained by jointly resampling blocks of training dates; record block length, horizon conversion, number of draws and PSD diagnostics. Freeze the radius or its inner-fold selection rule before evaluation. Equal holdings, execution timing, funded costs and hard constraints apply to both optimization arms. Recover the nominal objective at ρ=0; compare weights only for a unique optimum.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

In [ ]:
# Optional solver lab; invented annual decimal inputs and zero costs in both arms.
from importlib.util import find_spec

if all(find_spec(package) for package in ("numpy", "cvxpy", "clarabel", "matplotlib")):
    import cvxpy as cp
    import numpy as np
    import matplotlib.pyplot as plt

    mu_demo = np.array([0.035, 0.065, 0.085])
    # A.T @ A is return covariance; B.T @ B is uncertainty of estimated means.
    A = np.diag([0.05, 0.12, 0.22])
    B = np.diag([0.006, 0.014, 0.045])

    def solve_demo(cap, radius=None):
        weights = cp.Variable(3)
        risk_limit = cp.norm(A @ weights, 2) <= cap
        expected = mu_demo @ weights
        score = expected if radius is None else expected - radius * cp.norm(B @ weights, 2)
        problem = cp.Problem(cp.Maximize(score),
                             [cp.sum(weights) == 1, weights >= 0, risk_limit])
        problem.solve(solver="CLARABEL", tol_gap_abs=1e-10, tol_feas=1e-10,
                      tol_gap_rel=1e-10)
        if problem.status != cp.OPTIMAL:
            raise ValueError(f"Unverified solver status: {problem.status}")
        w = weights.value
        assert abs(sum(w)-1) < 1e-7 and min(w) >= -1e-7
        assert np.linalg.norm(A @ w) <= cap + 1e-7
        return w, float(problem.value), float(risk_limit.dual_value)

    cap = 0.09
    classic_w, classic_value, shadow = solve_demo(cap)
    zero_w, zero_value, _ = solve_demo(cap, 0.0)
    assert np.allclose(classic_w, zero_w, atol=2e-5)
    assert abs(classic_value-zero_value) < 1e-8
    step = 1e-4
    slope = (solve_demo(cap+step)[1] - solve_demo(cap-step)[1]) / (2*step)
    assert np.isclose(shadow, slope, rtol=0.01, atol=1e-4)
    half_step = step / 2
    second_slope = (solve_demo(cap+half_step)[1] - solve_demo(cap-half_step)[1]) / (2*half_step)
    assert np.isclose(shadow, second_slope, rtol=0.01, atol=1e-4)
    assert abs(solve_demo(0.30)[2]) < 1e-6
    print("Nominal local marginal return per unit of volatility cap:", round(shadow, 4))

    fig, ax = plt.subplots(figsize=(7, 4))
    for radius, label in [(None, "Classical"), (1.0, "Robust")]:
        curve = [solve_demo(limit, radius)[0] for limit in np.linspace(.05, .16, 20)]
        ax.plot([np.linalg.norm(A @ w) for w in curve], [mu_demo @ w for w in curve],
                label=label)
        chosen = solve_demo(cap, radius)[0]
        ax.scatter(np.linalg.norm(A @ chosen), mu_demo @ chosen)
        print(label, "weights at the 9% cap:", np.round(chosen, 3))
    equal = np.full(3, 1/3)
    ax.scatter(np.linalg.norm(A @ equal), mu_demo @ equal, marker="x", label="Equal weight")
    ax.set(xlabel="Estimated annual volatility (decimal)", ylabel="Estimated annual return (decimal)",
           title="Synthetic allocations; dots mark a common risk cap")
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print("Install requirements-frontier.txt to run the optional solver and figure.")

### 6.2 Present defensible client options

**Evidence to retain:** eligible option and shortfall report. **Course reading:** [Guide Ultime Chapitre 16](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_16.md) · [Guide Ultime Chapitre 34](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_34.md).

#### 6.2.1 · Evaluate net mean–variance utility over the plausible phi interval with consistent horizon and decimal units

Evaluate net mean–variance utility over the plausible phi interval with consistent horizon and decimal units. Compare Sharpe against matching currency cash. Test an unresolved interval, tied alternatives and sensitivity to capacity.

**Procedure.** Evaluate Uφ=μnet−φσ²/2 at both endpoints per pair of portfolios; differences are affine in φ. Compute internal crossover 2Δμ/Δvariance when defined. Match annual units and currency to the elicitation.

**Verification and failure handling.** Report robust winner, crossover, ties or unresolved. For zero excess-return volatility mark Sharpe undefined. An unbounded/unreliable φ interval cannot justify a precise personal ranking.

**Bias and control.** Sleeve/household utility mismatch. Define the wealth base for returns, risk and elicited φ. With deterministic reserve and risky fraction a, total excess mean scales by a and variance by a². Recompute rankings at both bases and test transfer of elicited preferences before personalized ranking.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

In [ ]:
# Same phi applied to sleeve versus total wealth. The protected cash earns zero.
def utility(mean, variance, phi):
    return mean - phi * variance / 2
phi, risky_share = 4, 0.5
options = {"moderate": (0.03,0.0025), "volatile": (0.05,0.0225)}
def preferred(share):
    return max(options, key=lambda k: utility(share*options[k][0], share**2*options[k][1],phi))
assert preferred(1) == "moderate"
# At 50% the two are tied; at 40% the investment ranking reverses.
assert isclose(utility(.5*.03,.5**2*.0025,phi),utility(.5*.05,.5**2*.0225,phi))
assert preferred(0.4) == "volatile"

#### 6.2.2 · Display historical outcome, forecast range, shock loss, liquidity, fees and goal shortfall for each eligible option

Display historical outcome, forecast range, shock loss, liquidity, fees and goal shortfall for each eligible option. Explain whether a longer horizon, more savings or a revised target closes a gap. Verify a desired return cannot override a hard constraint.

**Procedure.** Translate each admissible allocation into investable capital, dated loss scenario, modeled fee, funding path and return range. Separate observed history, future scenario and realized prospective outcomes.

**Verification and failure handling.** Recalculate an example independently in spending currency. Report any goal gap and compare adjustments to contributions, horizon or target at the same capacity.

**Bias and control.** False precision in long-horizon goals. Simulate dated cash flows with resampled joint returns and FX, protect required withdrawals at their due dates, and vary inflation and sequence of returns. Report both frequency and size of funding shortfalls with assumptions.

**Unmodeled joint tail losses.** Start with empirical joint stress and block-resampled asset/FX paths at the withdrawal date. If the decision is sensitive to losses beyond the sample, register peaks-over-threshold/GPD as a sensitivity, check threshold and shape stability, exceedance clustering and uncertainty. A t-copula extension requires dated marginal fits, residual dependence diagnostics and comparison with empirical joint losses. With sparse extremes present scenario losses and sampling limits. Never convert a hypothetical stress into a probability without calibration.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Reverse withdrawal stress (guide 05, §5.13.3).** At the due date compute available cash plus saleable positions after FX, settlement and modeled fees. Start with the unacceptable event (available resources below amount due) and find the boundary along a specified shock path. Report exact coverage at equality, a pre-existing shortfall, or no reachable shortfall separately. Compare CHF 15,000 fully reserved against a larger or earlier payment. For joint shocks disclose how distance and feasibility are defined. The boundary carries no probability by itself.

**Client display.** Separate dated historical results, conditional monthly ranges and multi-year funding scenarios. Compare only feasible alternatives. When utility rankings flip across plausible tolerance or model inputs, present the trade-off and allow the client to choose. Use a practical-equivalence label only if the prespecified margin and uncertainty support it; a wide interval leads to an inconclusive label.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

In [ ]:
# At the due date, only cash and positions available for settlement can fund payment.
def first_shortfall_boundary(cash, saleable_chf, due):
    if min(cash, saleable_chf) < 0 or due <= 0:
        raise ValueError("Invalid funding inputs.")
    if cash >= due:
        return "covered for all sleeve losses", None
    if cash + saleable_chf < due:
        return "already underfunded", None
    return "shortfall above loss fraction", 1 - (due-cash)/saleable_chf

assert first_shortfall_boundary(15_000, 35_000, 15_000)[1] is None
status, boundary = first_shortfall_boundary(15_000, 35_000, 22_000)
assert status == "shortfall above loss fraction" and isclose(boundary, 0.8)
assert first_shortfall_boundary(15_000, 0, 22_000)[0] == "already underfunded"
print("CHF 22,000 due: payment is exactly funded at an 80% sleeve loss;",
      "a larger loss creates a shortfall under these settlement assumptions.")

## 7. Backtest and decide catalogue admission

In [ ]:
# A fixed initial charge precedes the holding period.
final_value = (100 - 0.20) * 1.10
assert isclose(final_value, 109.78)

### 7.1 Replay realistic transactions

**Evidence to retain:** order, fill, cash and cost ledgers. **Course reading:** [guide couts transaction trading](Theory/Course_Notes/guide_couts_transaction_trading.md) · [Guide Ultime Chapitre 06](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_06.md).

#### 7.1.1 · At every origin save eligible data, forecast, signal and intended order before the execution timestamp

At every origin save eligible data, forecast, signal and intended order before the execution timestamp. Execute at the first eligible quote under the registered rule. Verify a next-close order earns no return before its fill.

**Procedure.** Create decision and fill tables with origin, data cutoff, intended weights, quote, executable timestamp, fill status and first holding interval. Store orders before outcome evaluation.

**Verification and failure handling.** Move every post-decision observation and verify past orders are invariant. Check a next-close buy manually and a skipped fill with unchanged units.

**Bias and control.** Unfillable signals. Simulate a missed or partial fill and an unavailable quote. Report attempted versus filled orders and delay. No profit accrues on unfilled units.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

In [ ]:
# Signal at 100, fill next close at 110, sale at 121.
signal_price, fill, exit_price = 100, 110, 121
units = 100 / fill
assert isclose(units*exit_price, 110)
assert not isclose(units*exit_price, 100*exit_price/signal_price)

#### 7.1.2 · Maintain shares, cash, FX, reserves, external flows and costs

Maintain shares, cash, FX, reserves, external flows and costs. Charge on amounts traded from drifted weights. Reconcile gross and net wealth against hand examples for initial purchase, no trade, sale and withdrawal.

**Procedure.** Ledger identities: wealth=cash+Σunits×price×FX; cash_after=cash_before+external_flow−Σsigned_trade_value−fees. Estimate half-spread on gross traded notional; report one-way turnover convention separately. Use bid/ask fills or spread model once.

**Verification and failure handling.** Reconcile each fill to currency rounding tolerance, cash availability and fees. Compare zero-cost, positive-cost, no-trade and restricted-liquidity withdrawal paths.

**Bias and control.** Double counting spread, management fees or FX. Compare mid-plus-explicit-cost and bid/ask fill ledgers on a known trade. Record embedded versus additional charges per instrument.

**Liquidity and implementation bias.** Compare planned trade notional with observable lagged volume, timestamped spread and settlement. When available, report decision-to-fill implementation shortfall and opportunity cost on unfilled amounts. If only closing prices exist, stress participation limits, slippage and partial fills and label costs modeled. Archive the wealth/withdrawal sensitivity, including unfilled orders.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

#### 7.1.3 · Compare buy-and-hold and simple rebalancing over the same dates

Compare buy-and-hold and simple rebalancing over the same dates. Stress costs, calendars, crises, estimation windows and missed trades. Report net wealth, drawdown, turnover, risk breaches and withdrawal shortfalls.

**Procedure.** Run a walk-forward panel of strategies, comparisons and cost levels on matching dates. Stress warm-up, FX, execution lag, spread widening and missed trades with prespecified variants.

**Verification and failure handling.** Report paired terminal wealth, annualized return, max drawdown, ES, turnover, cash shortfall and breach duration. A result that reverses under plausible cost or date choices is flagged.

**Bias and control.** Path dependence. Keep the same return paths and cash flows across strategies and include an adverse ordering before withdrawal. Report the difference between terminal wealth and cash available on the exact due date.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

### 7.2 Decide what may reach the application

**Evidence to retain:** study disposition and signed catalogue version. **Course reading:** [Guide Ultime Chapitre 22](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_22.md) · [Guide Ultime Chapitre 30](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_30.md).

#### 7.2.1 · Review frozen out-of-sample results against the registered economic threshold and paired uncertainty

Review frozen out-of-sample results against the registered economic threshold and paired uncertainty. Label retrospective results. Record retain, reject or descriptive-only for every study with a reason.

**Procedure.** Compare paired net outcome and its block interval with the predeclared minimum useful gain and admissibility constraints. Distinguish development, retrospective and genuinely prospective evidence.

**Verification and failure handling.** Produce one retain/reject/inconclusive record per variant including failures and reasons. Do not reuse the final period after changing a rule.

**Bias and control.** Absence-of-evidence error. Compare the interval with the material-gain threshold. Claim practical equivalence only with prespecified equivalence margins and suitable interval/test; wide uncertainty remains inconclusive.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Paired client-value test (guides 01 V18 and 04 §4.15.5).** Register one primary horizon, φ range, economic gain threshold and fallback before outcomes. Archive both frontiers and weights at each common origin. Compare net mean–variance scores on realized returns with the same household wealth base and horizon as φ. Recompute mean and variance per strategy inside each paired date-block draw. Report Δscore and intervals, turnover, costs, risk breaches and uncovered cash at due dates. Label this as mean–variance utility; an exact certainty equivalent requires a specified utility and inversion. Display dated estimated frontiers on common axes and realized comparisons in a separate table. Show sensitivity across prespecified windows, covariance estimators and fee levels. Record inconclusive results and keep the simpler benchmark when evidence is insufficient.

**Allocation experiment.** Keep the mandate and permitted information fixed; compare fixed mix, classical and robust frontiers over the same origins. Evaluate paired net utility, shortfall at withdrawal, breaches and turnover. With fictional mandates label results as conditional on those mandates. Retain the admissible baseline if the improvement is too small or inconclusive. Record questionnaire validation separately.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

#### 7.2.2 · Independently reproduce admitted catalogue entries, costs and mandate screens

Independently reproduce admitted catalogue entries, costs and mandate screens. Attach as-of date, uncertainty, fallback and data permissions. Check that a failed study cannot silently enter a recommendation.

**Procedure.** Freeze catalogue rows with study version, eligible mandate class, currency, date, benchmark, fee assumptions, forecast expiry, permissions and fallback. Rebuild from immutable source IDs.

**Verification and failure handling.** Have a reviewer rerun a selected row and a rejected row. Only approved, unexpired records pass the app eligibility query.

**Bias and control.** Selective promotion. Include rejected and expired entries with reason codes. Verify that changing the candidate cannot inherit a prior candidate's holdout status.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

## 8. Monitor and review in the web app

In [ ]:
# End-of-period withdrawal after valuation at an unchanged price.
start, withdrawal = 100, 15
closing_value = 85
investment_return = (closing_value + withdrawal) / start - 1
assert isclose(investment_return, 0)

### 8.1 Value the selected allocation

**Evidence to retain:** holdings snapshot and valuation reconciliation. **Course reading:** [Guide Ultime Chapitre 07](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_07.md) · [Guide Ultime Chapitre 26](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_26.md).

#### 8.1.1 · Persist confirmed mandate, allocation, units, cash, model version and selection date

Persist confirmed mandate, allocation, units, cash, model version and selection date. Connect an authorized close/FX feed with timestamps, refresh policy and stale-data handling. Test missing, delayed and corrected quotes.

**Procedure.** Use versioned holdings and append-only close/FX observations, provider timestamp, retrieval time, calendar and correction lineage. Store model and consent versions at selection.

**Verification and failure handling.** Test missing market day versus holiday, stale FX with fresh asset prices and vendor correction. Display last reliable valuation and withhold actionable proposals on stale inputs.

**Bias and control.** Feed survivorship and revision. Preserve as-received and subsequently corrected valuations. Report unavailable valuation days and elapsed quote age.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

#### 8.1.2 · Chart total value, flows, net time-weighted performance, drawdown, drift and goal progress

Chart total value, flows, net time-weighted performance, drawdown, drift and goal progress. Show last refresh and distinguish projections. Test that a withdrawal reduces value without creating a fictitious investment loss.

**Procedure.** For end-period cash flow F compute subperiod return (V_end−F)/V_start−1, chain subperiods for TWR. Split at actual flows when timestamps allow. Document timing assumptions where unavailable.

**Verification and failure handling.** Use a zero-market-return withdrawal and a deposit during a price move. Reconcile chart change into market P&L, FX, fees and external flows.

**Bias and control.** Cash-flow timing bias. Reconcile a same-day deposit before versus after a price change; disclose when only end-of-day flows permit an approximation.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

### 8.2 Decide whether to rebalance

**Evidence to retain:** hold/trade comparison and confirmation events. **Course reading:** [Guide Ultime Chapitre 21](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_21.md) · [guide couts transaction trading](Theory/Course_Notes/guide_couts_transaction_trading.md).

#### 8.2.1 · Trigger review after material drift, changed risk, a due expense or a revised mandate

Trigger review after material drift, changed risk, a due expense or a revised mandate. Compare hold versus trade on identical assumptions. Show turnover, costs, liquidity and expected effect on the goal.

**Procedure.** Run hold and proposed trade from identical current holdings, timestamps, cash forecasts and costs. Estimate post-trade risk drift and net change in withdrawal coverage.

**Verification and failure handling.** Compare accepted, declined and hold outcomes without attributing selection effects to the intervention. Allow hold when benefit is within estimation or cost uncertainty.

**Bias and control.** Self-selection into trades. Compute a dated shadow hold path for every eligible proposal with identical flows and executable prices. Describe acceptance-conditioned outcomes separately; attribution to the intervention needs a credible comparison design.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Hold or trade.** Freeze the materiality margin, costs and uncertainty rule before testing. For a discretionary rebalance, keep the current eligible allocation if the paired net benefit is inconclusive. A new hard constraint or withdrawal triggers a funding review even if expected return barely changes; show the least disruptive feasible correction, or the uncovered amount if none exists. Request confirmation.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

In [ ]:
# Toy decision rule: intervals are for paired net gains after modeled costs.
def review_action(valuation_current, mandate_breached, overlay_admitted, net_gain, margin):
    if not valuation_current:
        return "refresh valuation"
    if mandate_breached:
        return "review funding and feasibility"
    if not overlay_admitted:
        return "evaluate registered baseline"
    low, high = net_gain
    if high < low or margin <= 0:
        raise ValueError("Invalid interval or materiality margin")
    if low > margin:
        return "present proposal for confirmation"
    if low >= -margin and high <= margin:
        return "hold: within practical margin"
    if high < -margin:
        return "hold: trade is worse"
    return "hold: evidence inconclusive"

assert review_action(False, True, False, None, .002) == "refresh valuation"
assert review_action(True, True, False, None, .002) == "review funding and feasibility"
assert review_action(True, False, False, None, .002) == "evaluate registered baseline"
assert review_action(True, False, True, (-.01,.03), .002) == "hold: evidence inconclusive"
assert review_action(True, False, True, (-.001,.001), .002) == "hold: within practical margin"
assert review_action(True, False, True, (.003,.01), .002) == "present proposal for confirmation"

#### 8.2.2 · Require explicit client confirmation and log declines, holds and accepted changes

Require explicit client confirmation and log declines, holds and accepted changes. Suspend proposals on stale inputs. Test double confirmation, inability to sell, settlement delay and insufficient liquidity.

**Procedure.** Persist review ID, mandate version and decision before applying one simulated trade. Treat repeated confirmations idempotently and revalue if the quote or mandate changes.

**Verification and failure handling.** Test duplicate confirmation, expired proposal, delayed settlement and insufficient available cash. Reconcile the final units to a single confirmed execution.

**Bias and control.** Repeated-action bias. Report unique proposals, stale expiries, declines and duplicate confirmations. Trace one decision to one execution or a documented no-trade outcome.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

In [ ]:
# Replaying the same confirmation must not apply the order twice.
applied = set()
cash = 100
for proposal_id, amount in [("demo-1", 15), ("demo-1", 15)]:
    if proposal_id not in applied:
        cash -= amount
        applied.add(proposal_id)
assert cash == 85 and len(applied) == 1

## 9. Verify, present and release

### 9.1 Demonstrate the complete workflow

**Evidence to retain:** release evidence and client pitch trace. **Course reading:** [Guide Ultime Chapitre 07](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_07.md) · [Guide Ultime Chapitre 30](Theory/Methodology/Chapter_Guides_00-34/Guide_Ultime_Chapitre_30.md).

#### 9.1.1 · Run unit, integration and mobile/desktop browser checks

Run unit, integration and mobile/desktop browser checks. Exercise uncertain profile, infeasible goal, missing private files, stale feeds and the full confirmed journey. Store commands, outcomes and known limitations.

**Procedure.** Create end-to-end fixtures linking participant events, mandate, as-of market snapshot, order and dashboard. Run pytest plus mobile/desktop browser smoke tests with a documented seed.

**Verification and failure handling.** Save expected/actual amounts and screenshots of uncertain, infeasible and stale states. A skipped integration test stays explicitly unverified.

**Bias and control.** Happy-path testing. Include failed imports, timeouts and incomplete profiles in test denominators. Allocate independent fixtures to developer tests and acceptance tests.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Counterfactual acceptance.** Replay one confirmed mandate twice: change only tolerance, then only withdrawal date. Fail release if tolerance moves reserves or a due-date change skips liquidity review. Also simulate a failed signal, a stale valuation and near-tied options; verify the fallback, suspension and unresolved-choice displays respectively.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

#### 9.1.2 · Reproduce public examples in a fresh clone

Reproduce public examples in a fresh clone. Review source/display rights, consent, sensitive profile storage, deletion and secrets. Verify only intended demo inputs are distributed.

**Procedure.** Create a redistributable input manifest, environment lock, run command and permissions table covering data, course texts and participant records. Test synthetic route in a network-free checkout.

**Verification and failure handling.** Compare expected hashes/metrics. Check exports for private fields and verify deletion on a synthetic participant.

**Bias and control.** Irreproducible provenance. Compare a fresh-clone input/version manifest to the approved one. Record private components the public reviewer cannot reproduce.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.

#### 9.1.3 · Prepare the 15-minute client pitch around one mandate, two feasible alternatives or a documented shortfall, and a dated review

Prepare the 15-minute client pitch around one mandate, two feasible alternatives or a documented shortfall, and a dated review. Cross-check figures against the notebook, course requirements against 1.1.1 and release decisions against the journal.

**Procedure.** Trace each reported number to study ID, sample dates, fee convention and notebook output. Use a client scenario to demonstrate uncertainty and a dated review decision.

**Verification and failure handling.** Rehearse to the course time limit. Have a second reader reproduce one slide number and identify where the scenario could fail.

**Bias and control.** Presentation cherry-picking. Choose the demo scenario before inspecting its performance and show a failure or infeasible case alongside it. Trace every headline result to its full denominator.

**Bias evidence** · Population/denominator: pending · Expected direction: pending · Control result and effect on client decision: pending · Residual uncertainty: pending.

**Run record** · Input/vintage: pending · Command/configuration: pending · Observed statistic, sample size and uncertainty: pending · Exceptions and decision: pending.